In [1]:
!pip -q install sentence-transformers faiss-cpu implicit pyarrow

In [2]:
import os
import re
import json
import time
import pickle
import zipfile

import numpy as np
import pandas as pd
import faiss
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/step13")

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("===== ENVIRONMENT =====")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU NAME:",
        torch.cuda.get_device_name(0)
    )

print("FAISS:", faiss.__version__)
print("OUTPUT:", OUT_DIR)

===== ENVIRONMENT =====
PyTorch: 2.10.0+cu128
GPU: True
GPU NAME: Tesla T4
FAISS: 1.15.1
OUTPUT: /kaggle/working/step13


In [3]:
REQUIRED_FILES = [
    "unified_products.parquet",
    "unified_reviews.parquet",
    "product_faiss_moi.index",
    "product_embedding_metadata.parquet",
    "step11_config.json",
    "bpr_model.pkl",
    "cf_mappings.pkl",
    "step12_config.json",
]

found_files = {}

# Tìm file đã được Add Input trực tiếp
for filename in REQUIRED_FILES:
    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if matches:
        found_files[filename] = matches[0]

# Nếu còn thiếu, tìm trong các ZIP Step 10–12
missing = [
    name
    for name in REQUIRED_FILES
    if name not in found_files
]

if missing:
    EXTRACT_DIR = (
        Path("/kaggle/working")
        / "step13_inputs"
    )

    EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    for zip_path in INPUT_DIR.rglob("*.zip"):
        if zip_path.name not in {
            "step10_final.zip",
            "step11_final.zip",
            "step12_final.zip",
        }:
            continue

        target = (
            EXTRACT_DIR
            / zip_path.stem
        )

        target.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)

    for filename in missing:
        matches = list(
            EXTRACT_DIR.rglob(filename)
        )

        if matches:
            found_files[filename] = matches[0]

print("===== STEP 13 INPUT FILES =====")

for filename in REQUIRED_FILES:
    path = found_files.get(filename)

    if path is None:
        print("MISSING:", filename)
    else:
        print("FOUND:", filename, "→", path)

still_missing = [
    name
    for name in REQUIRED_FILES
    if name not in found_files
]

assert not still_missing, (
    "Thiếu file đầu vào: "
    + ", ".join(still_missing)
)

print("\nPASS: Đã tìm thấy toàn bộ file đầu vào.")

===== STEP 13 INPUT FILES =====
FOUND: unified_products.parquet → /kaggle/input/datasets/trnquangtriu/13dataset/unified_products.parquet
FOUND: unified_reviews.parquet → /kaggle/input/datasets/trnquangtriu/13dataset/unified_reviews.parquet
FOUND: product_faiss_moi.index → /kaggle/input/datasets/trnquangtriu/13dataset/product_faiss_moi.index
FOUND: product_embedding_metadata.parquet → /kaggle/input/datasets/trnquangtriu/13dataset/product_embedding_metadata.parquet
FOUND: step11_config.json → /kaggle/input/datasets/trnquangtriu/13dataset/step11_config.json
FOUND: bpr_model.pkl → /kaggle/input/datasets/trnquangtriu/13dataset/bpr_model.pkl
FOUND: cf_mappings.pkl → /kaggle/input/datasets/trnquangtriu/bosung/cf_mappings.pkl
FOUND: step12_config.json → /kaggle/input/datasets/trnquangtriu/13dataset/step12_config.json

PASS: Đã tìm thấy toàn bộ file đầu vào.


In [4]:
products = pd.read_parquet(
    found_files["unified_products.parquet"]
)

reviews = pd.read_parquet(
    found_files["unified_reviews.parquet"]
)

print("PRODUCTS:", products.shape)
print("REVIEWS:", reviews.shape)

assert len(products) == 164_926
assert len(reviews) == 767_179

assert products["product_id"].is_unique

print("\nPRODUCT COLUMNS:")
print(products.columns.tolist())

print("\nREVIEW COLUMNS:")
print(reviews.columns.tolist())

print("\nREVIEWS BY SOURCE:")

if "source" in reviews.columns:
    display(
        reviews["source"].value_counts(
            dropna=False
        )
    )

print("\nSAMPLE REVIEWS:")

display(
    reviews.head(3)
)

PRODUCTS: (164926, 60)
REVIEWS: (767179, 16)

PRODUCT COLUMNS:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'smart_product_name', 'smart_source', 'smart_currency', 'smart_price', 'smart_rating', 'smart_review_count', 'product_family', 'product_type', 'classification_method', 'matched_rule', 'has_valid_price', 'has_valid_currency', 'price_filter_eligible', 'quality_issue_count', 'quality_flags_json', 'product_family_v1', 'product_type_v1', 'classification_method_v1', 'matched_rule_v1', 'product_family_v2', 'product_type_v2', 'classification_method_v2', 'classification_reason_v2', 'etl_review_rows', 'etl_review_text_rows', 'etl_review_evidence_rows', 'has_review_evidence', 'product_url_available', 'product_type_candidate', 'product_family_candidate', 'classification_requires_validation', 'top5_evidence_ready', 'asin', 'meta_title', 'm

source
amazon    736093
shopee     31086
Name: count, dtype: int64


SAMPLE REVIEWS:


,review_id,user_id,product_id,rating,review_title,review_text,timestamp,verified_purchase,helpful_vote,language,source,smart_review_text,smart_review_rating,smart_source,has_review_evidence,review_evidence_eligible
0,review_0000000,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B07N69T6TM,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,1592678549731,True,0.0,en,amazon,These didn’t work. Idk if they were damaged in...,1.0,amazon,True,True
1,review_0000001,amazon_AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,amazon_B01G8JO5F2,5.0,Excellent!,I love these. They even come with a carry case...,1523093017534,True,0.0,en,amazon,I love these. They even come with a carry case...,5.0,amazon,True,True
2,review_0000002,amazon_AG2L7H23R5LLKDKLBEF2Q3L2MVDA,amazon_B07CJYMRWM,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,1676601581238,True,0.0,en,amazon,I've bought these headphones three times becau...,5.0,amazon,True,True


In [5]:
assert "product_id" in reviews.columns

product_ids = set(
    products["product_id"]
)

review_product_ids = set(
    reviews["product_id"].dropna()
)

matched_review_products = (
    product_ids
    &
    review_product_ids
)

print(
    "TOTAL PRODUCTS:",
    len(product_ids)
)

print(
    "PRODUCTS WITH LINKED REVIEWS:",
    len(matched_review_products)
)

print(
    "REVIEW PRODUCT IDS NOT IN PRODUCTS:",
    len(
        review_product_ids - product_ids
    )
)

print(
    "PRODUCT COVERAGE WITH REVIEWS:",
    round(
        len(matched_review_products)
        /
        len(product_ids)
        * 100,
        2
    ),
    "%"
)

TOTAL PRODUCTS: 164926
PRODUCTS WITH LINKED REVIEWS: 123351
REVIEW PRODUCT IDS NOT IN PRODUCTS: 0
PRODUCT COVERAGE WITH REVIEWS: 74.79 %


In [6]:
hybrid_matches = list(
    INPUT_DIR.rglob(
        "hybrid_top5_test.parquet"
    )
)

if hybrid_matches:
    hybrid_test = pd.read_parquet(
        hybrid_matches[0]
    )

    top5_ids = (
        hybrid_test["product_id"]
        .head(5)
        .tolist()
    )

else:
    test_path = list(
        INPUT_DIR.rglob(
            "step12_test_results.json"
        )
    )

    assert test_path, (
        "Không tìm thấy kết quả "
        "kiểm thử Step 12."
    )

    with open(
        test_path[0],
        "r",
        encoding="utf-8"
    ) as f:
        step12_tests = json.load(f)

    top5_ids = (
        step12_tests[0]["product_ids"][:5]
    )

print("TOP 5 PRODUCT IDS:")

for pid in top5_ids:
    print("-", pid)

top5_reviews = reviews[
    reviews["product_id"].isin(
        top5_ids
    )
].copy()

print(
    "\nTOTAL LINKED REVIEWS:",
    len(top5_reviews)
)

print("\nREVIEWS PER PRODUCT:")

review_counts = (
    top5_reviews
    .groupby("product_id")
    .size()
    .reindex(
        top5_ids,
        fill_value=0
    )
)

display(
    review_counts.rename(
        "linked_reviews"
    ).reset_index()
)

TOP 5 PRODUCT IDS:
- amazon_B073JWXGNT
- amazon_B011BRUOMO
- amazon_B0C2VMSBGX
- amazon_B016A5NH6C
- amazon_B0BVVJM2X8

TOTAL LINKED REVIEWS: 1666

REVIEWS PER PRODUCT:


,product_id,linked_reviews
0,amazon_B073JWXGNT,302
1,amazon_B011BRUOMO,1354
2,amazon_B0C2VMSBGX,1
3,amazon_B016A5NH6C,4
4,amazon_B0BVVJM2X8,5


In [7]:
possible_text_columns = [
    "review_text",
    "clean_review",
    "text",
    "content",
    "review_body",
    "body",
]

available_text_columns = [
    col
    for col in possible_text_columns
    if col in reviews.columns
]

print(
    "AVAILABLE REVIEW TEXT COLUMNS:",
    available_text_columns
)

print("\nREVIEW COLUMNS:")
print(reviews.columns.tolist())

for col in available_text_columns:

    nonempty = (
        reviews[col]
        .astype("string")
        .str.strip()
        .notna()
        &
        reviews[col]
        .astype("string")
        .str.strip()
        .ne("")
    ).fillna(False)

    print(
        col,
        "| nonempty:",
        int(nonempty.sum())
    )

AVAILABLE REVIEW TEXT COLUMNS: ['review_text']

REVIEW COLUMNS:
['review_id', 'user_id', 'product_id', 'rating', 'review_title', 'review_text', 'timestamp', 'verified_purchase', 'helpful_vote', 'language', 'source', 'smart_review_text', 'smart_review_rating', 'smart_source', 'has_review_evidence', 'review_evidence_eligible']
review_text | nonempty: 766776


In [8]:
import numpy as np
import pandas as pd

reviews_rag = reviews.copy()

INVALID_REVIEW_TEXT = {
    "",
    "none",
    "null",
    "nan",
    "n/a",
}

def clean_review_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if text.lower() in INVALID_REVIEW_TEXT:
        return ""

    return " ".join(text.split())


reviews_rag["original_review_text"] = (
    reviews_rag["review_text"]
    .apply(clean_review_text)
)

reviews_rag["rag_review_text"] = (
    reviews_rag["smart_review_text"]
    .apply(clean_review_text)
)

empty_smart = (
    reviews_rag["rag_review_text"]
    .eq("")
)

reviews_rag.loc[
    empty_smart,
    "rag_review_text"
] = reviews_rag.loc[
    empty_smart,
    "original_review_text"
]

reviews_rag = reviews_rag[
    reviews_rag["rag_review_text"].ne("")
].copy()

reviews_rag = reviews_rag.reset_index(
    drop=True
)

print("REVIEWS READY FOR RAG:", len(reviews_rag))

print(
    "UNIQUE REVIEW IDS:",
    reviews_rag["review_id"].nunique()
)

assert reviews_rag["review_id"].is_unique

display(
    reviews_rag[
        [
            "review_id",
            "product_id",
            "rating",
            "rag_review_text"
        ]
    ].head(3)
)

REVIEWS READY FOR RAG: 766696
UNIQUE REVIEW IDS: 766696


,review_id,product_id,rating,rag_review_text
0,review_0000000,amazon_B07N69T6TM,1.0,These didn’t work. Idk if they were damaged in...
1,review_0000001,amazon_B01G8JO5F2,5.0,I love these. They even come with a carry case...
2,review_0000002,amazon_B07CJYMRWM,5.0,I've bought these headphones three times becau...


In [9]:
reviews_by_product = {
    product_id: group.reset_index(drop=True)
    for product_id, group in reviews_rag.groupby(
        "product_id",
        sort=False
    )
}

def get_product_reviews(product_id):
    result = reviews_by_product.get(
        product_id
    )

    if result is None:
        return reviews_rag.iloc[0:0].copy()

    return result.copy()


print(
    "PRODUCTS WITH RAG REVIEWS:",
    len(reviews_by_product)
)

for product_id in top5_ids:
    product_reviews = get_product_reviews(
        product_id
    )

    print(
        product_id,
        "→",
        len(product_reviews),
        "reviews"
    )

PRODUCTS WITH RAG REVIEWS: 123304
amazon_B073JWXGNT → 302 reviews
amazon_B011BRUOMO → 1354 reviews
amazon_B0C2VMSBGX → 1 reviews
amazon_B016A5NH6C → 4 reviews
amazon_B0BVVJM2X8 → 5 reviews


In [10]:
import numpy as np

def retrieve_review_evidence(
    product_id,
    query,
    model,
    max_reviews=6,
    candidate_limit=300
):
    product_reviews = get_product_reviews(
        product_id
    )

    if product_reviews.empty:
        return []

    # Giới hạn số review cần encode cho mỗi sản phẩm
    if len(product_reviews) > candidate_limit:
        product_reviews = product_reviews.sample(
            n=candidate_limit,
            random_state=42
        ).reset_index(drop=True)

    texts = product_reviews[
        "rag_review_text"
    ].tolist()

    review_vectors = model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    scores = review_vectors @ query_vector

    product_reviews = product_reviews.copy()

    product_reviews["relevance_score"] = scores

    product_reviews["rating_numeric"] = (
        pd.to_numeric(
            product_reviews["rating"],
            errors="coerce"
        )
    )

    # Ưu tiên các review liên quan đến câu hỏi
    selected = (
        product_reviews
        .sort_values(
            "relevance_score",
            ascending=False
        )
        .head(max_reviews)
        .copy()
    )

    # Nếu có review đánh giá thấp, giữ ít nhất một
    # review để kiểm tra các nhận xét tiêu cực.
    negative_reviews = product_reviews[
        product_reviews["rating_numeric"].le(2)
    ].sort_values(
        "relevance_score",
        ascending=False
    )

    if not negative_reviews.empty:
        negative = negative_reviews.iloc[0]

        if negative["review_id"] not in set(
            selected["review_id"]
        ):
            selected = pd.concat(
                [
                    selected.head(
                        max_reviews - 1
                    ),
                    negative.to_frame().T
                ],
                ignore_index=True
            )

    evidence = []

    for _, row in selected.iterrows():
        evidence.append({
            "review_id": str(
                row["review_id"]
            ),
            "product_id": str(
                row["product_id"]
            ),
            "rating": (
                float(row["rating_numeric"])
                if pd.notna(
                    row["rating_numeric"]
                )
                else None
            ),
            "text": str(
                row["rag_review_text"]
            )[:1000],
            "relevance_score": float(
                row["relevance_score"]
            )
        })

    return evidence

In [11]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

review_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("MODEL:", MODEL_NAME)
print("DEVICE:", DEVICE)

print(
    "DIMENSION:",
    review_model.get_sentence_embedding_dimension()
)

MODEL: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
DEVICE: cuda
DIMENSION: 384


/tmp/ipykernel_267/288046707.py:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  review_model.get_sentence_embedding_dimension()


In [12]:
test_query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "dưới 15 USD để lưu video "
    "và sử dụng cho điện thoại"
)

all_evidence = {}

for product_id in top5_ids:

    evidence = retrieve_review_evidence(
        product_id=product_id,
        query=test_query,
        model=review_model,
        max_reviews=6
    )

    all_evidence[product_id] = evidence

    print("\n" + "=" * 70)
    print("PRODUCT:", product_id)

    print(
        "TOTAL AVAILABLE REVIEWS:",
        len(
            get_product_reviews(
                product_id
            )
        )
    )

    print(
        "SELECTED EVIDENCE:",
        len(evidence)
    )

    for item in evidence[:2]:
        print(
            "\nREVIEW ID:",
            item["review_id"]
        )

        print(
            "RATING:",
            item["rating"]
        )

        print(
            "TEXT:",
            item["text"][:350]
        )


PRODUCT: amazon_B073JWXGNT
TOTAL AVAILABLE REVIEWS: 302
SELECTED EVIDENCE: 6

REVIEW ID: review_0529082
RATING: 5.0
TEXT: I Bought this 32GB Micro SD Card because of its transfer speeds of up to 98MB/s**, Works great in my 12MP DSLR Camera. I have used SanDisk for years and have never had a problem. I highly recommend this product

REVIEW ID: review_0445991
RATING: 5.0
TEXT: SandDisk is a very reputable company. Purchased this 32GB because of a limitation on a device I am storing it it. If you don't have a size restriction, it isn't much more money to purchase a 64GB or even a 128 GB card (The later I have in my Samsung Galaxy cell phone), just remember to back up to a different storage media / drive / card so if anyth

PRODUCT: amazon_B011BRUOMO
TOTAL AVAILABLE REVIEWS: 1354
SELECTED EVIDENCE: 6

REVIEW ID: review_0070564
RATING: 5.0
TEXT: Nice sized memory card for a good price.

REVIEW ID: review_0661393
RATING: 1.0
TEXT: Definitely luxury to have this much space, and the card perf

In [13]:
import re
import numpy as np
import pandas as pd

def extract_capacities_gb(text):
    if pd.isna(text):
        return []

    text = str(text).lower()

    pattern = (
        r"(?<![\w.])"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(tb|gb)"
        r"\b"
    )

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    capacities = []

    for value, unit in matches:
        number = float(
            value.replace(",", ".")
        )

        if unit.lower() == "tb":
            number *= 1024

        capacities.append(number)

    return sorted(set(capacities))


tests = [
    "I bought this 32GB microSD card",
    "This is really a 64gb card",
    "It formats as 180 GB",
    "Transfer speeds up to 98MB/s",
    "32GB is cheaper than 64GB and 128GB",
]

for text in tests:
    print(
        text,
        "→",
        extract_capacities_gb(text)
    )

I bought this 32GB microSD card → [32.0]
This is really a 64gb card → [64.0]
It formats as 180 GB → [180.0]
Transfer speeds up to 98MB/s → []
32GB is cheaper than 64GB and 128GB → [32.0, 64.0, 128.0]


In [14]:
def check_review_capacity(
    review_text,
    expected_capacity_gb
):
    capacities = extract_capacities_gb(
        review_text
    )

    if not capacities:
        return "capacity_unspecified"

    conflicting = [
        capacity
        for capacity in capacities
        if abs(
            capacity - expected_capacity_gb
        ) > 0.001
    ]

    if conflicting:
        return "capacity_conflict"

    return "capacity_consistent"


test_cases = [
    ("I bought this 32GB card", 32),
    ("This is a 64GB card", 32),
    ("It formats as 180GB", 32),
    ("Works well in my phone", 32),
    ("32GB is cheaper than 64GB", 32),
]

for text, expected in test_cases:
    print(
        check_review_capacity(
            text,
            expected
        ),
        "→",
        text
    )

capacity_consistent → I bought this 32GB card
capacity_conflict → This is a 64GB card
capacity_conflict → It formats as 180GB
capacity_unspecified → Works well in my phone
capacity_conflict → 32GB is cheaper than 64GB


In [15]:
capacity_audit = []

for product_id in top5_ids:

    product_row = products[
        products["product_id"].eq(
            product_id
        )
    ]

    if product_row.empty:
        continue

    product_name = product_row.iloc[0][
        "product_name"
    ]

    product_capacities = (
        extract_capacities_gb(
            product_name
        )
    )

    if len(product_capacities) != 1:
        print(
            "SKIP AMBIGUOUS PRODUCT:",
            product_id,
            product_capacities
        )
        continue

    expected_capacity = (
        product_capacities[0]
    )

    product_reviews = (
        get_product_reviews(
            product_id
        )
    )

    statuses = product_reviews[
        "rag_review_text"
    ].apply(
        lambda text: check_review_capacity(
            text,
            expected_capacity
        )
    )

    counts = statuses.value_counts()

    capacity_audit.append({
        "product_id": product_id,
        "expected_capacity_gb": (
            expected_capacity
        ),
        "total_reviews": len(
            product_reviews
        ),
        "consistent": int(
            counts.get(
                "capacity_consistent",
                0
            )
        ),
        "unspecified": int(
            counts.get(
                "capacity_unspecified",
                0
            )
        ),
        "conflict": int(
            counts.get(
                "capacity_conflict",
                0
            )
        )
    })

capacity_audit_df = pd.DataFrame(
    capacity_audit
)

display(capacity_audit_df)

,product_id,expected_capacity_gb,total_reviews,consistent,unspecified,conflict
0,amazon_B073JWXGNT,32.0,302,8,289,5
1,amazon_B011BRUOMO,32.0,1354,23,1273,58
2,amazon_B0C2VMSBGX,32.0,1,0,1,0
3,amazon_B016A5NH6C,32.0,4,0,3,1
4,amazon_B0BVVJM2X8,32.0,5,0,5,0


In [16]:
def retrieve_review_evidence(
    product_id,
    query,
    model,
    max_reviews=6,
    candidate_limit=300,
    expected_capacity_gb=None
):
    product_reviews = (
        get_product_reviews(
            product_id
        )
    )

    if product_reviews.empty:
        return []

    product_reviews = (
        product_reviews.copy()
    )

    # 1. Kiểm tra dung lượng
    if expected_capacity_gb is not None:

        product_reviews[
            "capacity_status"
        ] = product_reviews[
            "rag_review_text"
        ].apply(
            lambda text: check_review_capacity(
                text,
                expected_capacity_gb
            )
        )

        product_reviews = (
            product_reviews[
                product_reviews[
                    "capacity_status"
                ].ne(
                    "capacity_conflict"
                )
            ].copy()
        )

    else:
        product_reviews[
            "capacity_status"
        ] = "not_checked"

    if product_reviews.empty:
        return []

    # 2. Giữ review nhất quán trước;
    # sau đó bổ sung review không nêu dung lượng.
    priority = {
        "capacity_consistent": 0,
        "capacity_unspecified": 1,
        "not_checked": 2
    }

    product_reviews[
        "_capacity_priority"
    ] = product_reviews[
        "capacity_status"
    ].map(
        priority
    ).fillna(3)

    product_reviews = (
        product_reviews.sort_values(
            [
                "_capacity_priority",
                "review_id"
            ]
        )
        .head(candidate_limit)
        .copy()
        .reset_index(drop=True)
    )

    # 3. Embedding review
    texts = product_reviews[
        "rag_review_text"
    ].tolist()

    review_vectors = model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)[0]

    product_reviews[
        "relevance_score"
    ] = (
        review_vectors @ query_vector
    )

    product_reviews[
        "rating_numeric"
    ] = pd.to_numeric(
        product_reviews["rating"],
        errors="coerce"
    )

    # 4. Chọn review liên quan nhất
    selected = (
        product_reviews
        .sort_values(
            [
                "relevance_score",
                "review_id"
            ],
            ascending=[
                False,
                True
            ]
        )
        .head(max_reviews)
        .copy()
    )

    # 5. Bổ sung tối đa một review
    # đánh giá thấp nếu có.
    negative_reviews = (
        product_reviews[
            product_reviews[
                "rating_numeric"
            ].le(2)
        ]
        .sort_values(
            "relevance_score",
            ascending=False
        )
    )

    if (
        max_reviews > 0
        and not negative_reviews.empty
    ):

        negative = (
            negative_reviews.iloc[0]
        )

        if negative["review_id"] not in set(
            selected["review_id"]
        ):

            selected = pd.concat(
                [
                    selected.head(
                        max_reviews - 1
                    ),
                    negative.to_frame().T
                ],
                ignore_index=True
            )

    # 6. Giữ provenance để LLM dẫn chứng
    evidence = []

    for _, row in selected.iterrows():

        evidence.append({
            "review_id": str(
                row["review_id"]
            ),
            "product_id": str(
                row["product_id"]
            ),
            "rating": (
                float(
                    row["rating_numeric"]
                )
                if pd.notna(
                    row["rating_numeric"]
                )
                else None
            ),
            "text": str(
                row["rag_review_text"]
            )[:1000],
            "capacity_status": str(
                row["capacity_status"]
            ),
            "relevance_score": float(
                row["relevance_score"]
            )
        })

    return evidence

In [17]:
all_evidence = {}

for product_id in top5_ids:

    product_row = products[
        products["product_id"].eq(
            product_id
        )
    ].iloc[0]

    capacities = extract_capacities_gb(
        product_row["product_name"]
    )

    expected_capacity = (
        capacities[0]
        if len(capacities) == 1
        else None
    )

    evidence = retrieve_review_evidence(
        product_id=product_id,
        query=test_query,
        model=review_model,
        max_reviews=6,
        expected_capacity_gb=(
            expected_capacity
        )
    )

    all_evidence[
        product_id
    ] = evidence

    print("\n" + "=" * 70)
    print("PRODUCT:", product_id)

    print(
        "EXPECTED CAPACITY:",
        expected_capacity
    )

    print(
        "SELECTED EVIDENCE:",
        len(evidence)
    )

    for item in evidence[:3]:

        print(
            "\nREVIEW:",
            item["review_id"]
        )

        print(
            "CAPACITY STATUS:",
            item["capacity_status"]
        )

        print(
            "RATING:",
            item["rating"]
        )

        print(
            "TEXT:",
            item["text"][:300]
        )


PRODUCT: amazon_B073JWXGNT
EXPECTED CAPACITY: 32.0
SELECTED EVIDENCE: 6

REVIEW: review_0529082
CAPACITY STATUS: capacity_consistent
RATING: 5.0
TEXT: I Bought this 32GB Micro SD Card because of its transfer speeds of up to 98MB/s**, Works great in my 12MP DSLR Camera. I have used SanDisk for years and have never had a problem. I highly recommend this product

REVIEW: review_0426199
CAPACITY STATUS: capacity_unspecified
RATING: 5.0
TEXT: Very good memory card

REVIEW: review_0561905
CAPACITY STATUS: capacity_unspecified
RATING: 5.0
TEXT: If you don't want to use your internal cell phone memory to store photos and videos, I recommend this product for you. The only concern is I noticed there is a little delay when taking the picture and storing in the memory. Not sure if this is phone setting or a memory card problem. Nonetheless, we 

PRODUCT: amazon_B011BRUOMO
EXPECTED CAPACITY: 32.0
SELECTED EVIDENCE: 6

REVIEW: review_0139542
CAPACITY STATUS: capacity_consistent
RATING: 5.0
TEXT: I 

In [18]:
VARIANT_WARNING_PATTERNS = [
    r"\bupgrade(?:d|s|ing)?\s+from\b",
    r"\bupgrading\s+from\b",
    r"\breplac(?:e|ed|ing)\b.{0,40}\b\d+\s*(?:gb|tb)\b",
    r"\b\d+\s*(?:gb|tb)\b.{0,50}\blimitation\b",
    r"\blimitation\b.{0,50}\b\d+\s*(?:gb|tb)\b",
    r"\bdoes\s+not\s+support\b.{0,50}\b\d+\s*(?:gb|tb)\b",
    r"\bdoesn't\s+support\b.{0,50}\b\d+\s*(?:gb|tb)\b",
]

def review_variant_warning(text):
    text = str(text).lower()

    for pattern in VARIANT_WARNING_PATTERNS:
        if re.search(pattern, text):
            return True

    return False


for review_id in [
    "review_0139542",
    "review_0375776",
    "review_0631333"
]:
    row = reviews_rag[
        reviews_rag["review_id"].eq(review_id)
    ]

    if row.empty:
        continue

    text = row.iloc[0]["rag_review_text"]

    print("\nREVIEW:", review_id)
    print("WARNING:", review_variant_warning(text))
    print("TEXT:", text[:500])


REVIEW: review_0139542
TEXT: I love the read & write speed of this micro SD card. I will NOT be returning this item even though I can NOT use this item. I purchased this card for my cell phone, apparently I have an older cell phone which does not support 64 gigs of space. So I recommend you google your cellphone and see how much space it actually supports. My LG Optimus phone will only support micro SD cards up to 32 gigabytes. So I have just ordered the $15 32gb SanDisk Micro SD Card because that is cheaper than a $120 cel

REVIEW: review_0375776
TEXT: This is my 2nd purchase in 2 years and it has proven to be great value for money and extending the storage capacity of mobile devices. I purchased this SD card at a bargain, almost half off the normal price. However, Microsoft eXFAT driver is required to overcome the existing 32GB limitation of some host devices.

REVIEW: review_0631333
TEXT: Great way to add extra storage to your device. I bought this to upgrade from a 32GB microSD ca

In [19]:
def build_product_evidence(
    product_id,
    query,
    max_reviews=6
):
    product_rows = products[
        products["product_id"].eq(product_id)
    ]

    if product_rows.empty:
        return None

    product = product_rows.iloc[0]

    capacities = extract_capacities_gb(
        product["product_name"]
    )

    expected_capacity = (
        capacities[0]
        if len(capacities) == 1
        else None
    )

    retrieved = retrieve_review_evidence(
        product_id=product_id,
        query=query,
        model=review_model,
        max_reviews=max_reviews * 2,
        expected_capacity_gb=expected_capacity
    )

    safe_evidence = []

    for item in retrieved:

        if review_variant_warning(item["text"]):
            continue

        safe_evidence.append(item)

        if len(safe_evidence) >= max_reviews:
            break

    total_reviews = len(
        get_product_reviews(product_id)
    )

    return {
        "product_id": product_id,
        "total_linked_reviews": total_reviews,
        "selected_evidence_count": len(
            safe_evidence
        ),
        "evidence": safe_evidence
    }

In [20]:
def safe_value(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if text.lower() in {
        "",
        "none",
        "null",
        "nan"
    }:
        return None

    return value


def build_rag_context(
    user_query,
    recommended_products,
    max_reviews_per_product=6
):
    context = {
        "user_query": user_query,
        "products": []
    }

    for _, ranked_row in (
        recommended_products.iterrows()
    ):

        product_id = ranked_row[
            "product_id"
        ]

        product_row = products[
            products["product_id"].eq(
                product_id
            )
        ]

        if product_row.empty:
            continue

        product = product_row.iloc[0]

        evidence_data = (
            build_product_evidence(
                product_id=product_id,
                query=user_query,
                max_reviews=(
                    max_reviews_per_product
                )
            )
        )

        product_info = {
            "product_id": product_id,

            "product_name": safe_value(
                product["product_name"]
            ),

            "source": safe_value(
                product["source"]
            ),

            "price": safe_value(
                product["price"]
            ),

            "currency": safe_value(
                product["currency"]
            ),

            "average_rating": safe_value(
                product["average_rating"]
            ),

            "review_count": safe_value(
                product["review_count"]
            ),

            "description": safe_value(
                product["description"]
            ),

            "product_features": safe_value(
                product["product_features"]
            ),

            "product_url": safe_value(
                product["product_url"]
            ),

            "total_linked_reviews": (
                evidence_data[
                    "total_linked_reviews"
                ]
            ),

            "selected_evidence_count": (
                evidence_data[
                    "selected_evidence_count"
                ]
            ),

            "review_evidence": (
                evidence_data["evidence"]
            )
        }

        context["products"].append(
            product_info
        )

    return context

In [21]:
test_query = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "dưới 15 USD để lưu video "
    "và sử dụng cho điện thoại"
)

# existing_user_result được tạo ở Step 12,
# nên trong notebook Step 13 ta đọc từ file
# hybrid_top5_test.parquet đã Add Input.

hybrid_matches = list(
    INPUT_DIR.rglob(
        "hybrid_top5_test.parquet"
    )
)

if hybrid_matches:

    recommended_top5 = pd.read_parquet(
        hybrid_matches[0]
    )

else:

    # Dùng product_id đã đọc từ
    # step12_test_results.json ở Cell 6.
    # Chỉ để kiểm tra RAG, không chạy lại Hybrid.
    recommended_top5 = pd.DataFrame({
        "product_id": top5_ids
    })

rag_context = build_rag_context(
    user_query=test_query,
    recommended_products=recommended_top5,
    max_reviews_per_product=6
)

print(
    "PRODUCTS IN CONTEXT:",
    len(rag_context["products"])
)

for item in rag_context["products"]:

    print("\n" + "=" * 65)

    print(
        "PRODUCT:",
        item["product_name"]
    )

    print(
        "PRICE:",
        item["price"],
        item["currency"]
    )

    print(
        "RATING:",
        item["average_rating"]
    )

    print(
        "LINKED REVIEWS:",
        item["total_linked_reviews"]
    )

    print(
        "SELECTED EVIDENCE:",
        item["selected_evidence_count"]
    )

    print(
        "PRODUCT URL:",
        item["product_url"]
    )

    print(
        "REVIEW IDS:",
        [
            evidence["review_id"]
            for evidence in item[
                "review_evidence"
            ]
        ]
    )

assert len(
    rag_context["products"]
) == 5

print(
    "\nPASS: RAG CONTEXT BUILT."
)

PRODUCTS IN CONTEXT: 5

PRODUCT: SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card with Adapter - 98MB/s, C10, U1, Full HD, A1, Micro SD Card - SDSQUAR-032G-GN6MA
PRICE: 8.66 USD
RATING: 4.7
LINKED REVIEWS: 302
SELECTED EVIDENCE: 6
PRODUCT URL: None
REVIEW IDS: ['review_0529082', 'review_0426199', 'review_0561905', 'review_0639235', 'review_0198424', 'review_0046578']

PRODUCT: SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-GN6MA)
PRICE: 8.99 USD
RATING: 4.6
LINKED REVIEWS: 1354
SELECTED EVIDENCE: 6
PRODUCT URL: None
REVIEW IDS: ['review_0139542', 'review_0070564', 'review_0389996', 'review_0037177', 'review_0025295', 'review_0083613']

PRODUCT: Hugolog 32GB Micro SD Card, MicroSDXC UHS-I Memory Card for Laview Camera - 100MB/s, 667X, U1, Class10, FHD Video V10, A1, FAT32, High Speed Flash TF Card P500 for Phone/Dash Cam/Tablet/PC/Computer
PRICE: 7.99 USD
RATING: 4.5
LINKED REVIEWS: 1
SELECTED EVIDENCE: 1
PRODUCT URL: None
REVIEW IDS: ['revi

In [22]:
!pip -q install -U transformers accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 74.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [23]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    total_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    print(
        "GPU MEMORY:",
        round(total_memory, 2),
        "GB"
    )

    print(
        "ALLOCATED:",
        round(
            torch.cuda.memory_allocated(0)
            / 1024**3,
            2
        ),
        "GB"
    )

CUDA: True
GPU: Tesla T4
GPU MEMORY: 14.56 GB
ALLOCATED: 0.45 GB


In [24]:
import importlib.metadata as md

packages = [
    "transformers",
    "huggingface-hub",
    "accelerate",
    "bitsandbytes",
    "torch",
]

for package in packages:
    try:
        print(
            package,
            "==",
            md.version(package)
        )
    except md.PackageNotFoundError:
        print(package, "NOT INSTALLED")

transformers == 5.17.0
huggingface-hub == 1.33.0
accelerate == 1.15.0
bitsandbytes == 0.50.2
torch == 2.10.0+cu128


In [25]:
import transformers
import huggingface_hub
import accelerate
import bitsandbytes

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from transformers.models.qwen2.configuration_qwen2 import (
    Qwen2Config
)

print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

config = Qwen2Config()

print("Qwen2Config:", type(config).__name__)

print("\nPASS: Qwen2 import thành công.")

Transformers: 4.51.3
HF Hub: 0.30.2
Accelerate: 1.6.0
BitsAndBytes: 0.50.2
Qwen2Config: Qwen2Config

PASS: Qwen2 import thành công.


In [26]:
%pip uninstall -y transformers huggingface-hub accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: huggingface_hub 1.33.0
Uninstalling huggingface_hub-1.33.0:
  Successfully uninstalled huggingface_hub-1.33.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2
Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install --no-cache-dir --force-reinstall --no-deps \
    "transformers==4.51.3" \
    "huggingface-hub==0.30.2" \
    "accelerate==1.6.0" \
    "bitsandbytes==0.45.5"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 195.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 356.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 333.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 168.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install -q \
    "tokenizers>=0.21,<0.22" \
    "safetensors>=0.4.3"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [29]:
import importlib.metadata as md

expected = {
    "transformers": "4.51.3",
    "huggingface-hub": "0.30.2",
    "accelerate": "1.6.0",
    "bitsandbytes": "0.45.5",
}

for package, version in expected.items():
    actual = md.version(package)

    print(f"{package}: {actual}")

    assert actual == version, (
        f"{package}: expected {version}, "
        f"got {actual}"
    )

print("\nPASS: Library versions are correct.")

transformers: 4.51.3
huggingface-hub: 0.30.2
accelerate: 1.6.0
bitsandbytes: 0.45.5

PASS: Library versions are correct.


In [30]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from transformers.models.qwen2.configuration_qwen2 import (
    Qwen2Config
)

from transformers.models.qwen2.modeling_qwen2 import (
    Qwen2ForCausalLM
)

print("Qwen2Config:", Qwen2Config.__name__)
print("Qwen2ForCausalLM:", Qwen2ForCausalLM.__name__)

print("\nPASS: Qwen model imports successfully.")

Qwen2Config: Qwen2Config
Qwen2ForCausalLM: Qwen2ForCausalLM

PASS: Qwen model imports successfully.


In [31]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle."

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

llm.eval()

print("PASS: Qwen LLM loaded.")
print("MODEL:", LLM_NAME)
print("DEVICE:", llm.device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

PASS: Qwen LLM loaded.
MODEL: Qwen/Qwen2.5-3B-Instruct
DEVICE: cuda:0


In [32]:
from pathlib import Path
import json

OUT_DIR = Path("/kaggle/working/step13")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_PATH = OUT_DIR / "rag_context_test.json"

if "rag_context" in globals():

    with open(
        CONTEXT_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            rag_context,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )

    print("PASS: Đã lưu RAG Context.")
    print("FILE:", CONTEXT_PATH)
    print("PRODUCTS:", len(rag_context["products"]))

else:

    print("Biến rag_context đã mất sau Restart Session.")

PASS: Đã lưu RAG Context.
FILE: /kaggle/working/step13/rag_context_test.json
PRODUCTS: 5


In [33]:
import transformers
import huggingface_hub
import accelerate
import bitsandbytes

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from transformers.models.qwen2.configuration_qwen2 import (
    Qwen2Config
)

print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

config = Qwen2Config()

print("Qwen2Config:", type(config).__name__)

print("\nPASS: Qwen2 import thành công.")

Transformers: 4.51.3
HF Hub: 0.30.2
Accelerate: 1.6.0
BitsAndBytes: 0.50.2
Qwen2Config: Qwen2Config

PASS: Qwen2 import thành công.


In [34]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

assert torch.cuda.is_available(), (
    "Cần bật GPU trong Kaggle."
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

llm.eval()

print("PASS: Qwen LLM loaded.")
print("MODEL:", LLM_NAME)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PASS: Qwen LLM loaded.
MODEL: Qwen/Qwen2.5-3B-Instruct


In [35]:
# CELL 26 — BUILD COMPACT LLM CONTEXT

import json
from pathlib import Path

CONTEXT_PATH = Path(
    "/kaggle/working/step13/rag_context_test.json"
)

with open(CONTEXT_PATH, "r", encoding="utf-8") as f:
    rag_context = json.load(f)

assert len(rag_context["products"]) > 0, (
    "RAG Context không có sản phẩm."
)

def shorten_text(value, max_chars=500):
    if value is None:
        return ""

    text = str(value).strip()

    if text.lower() in {"none", "nan", "null"}:
        return ""

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def compact_product(product):

    compact = {}

    # Giữ nguyên tên trường từ RAG Context.
    # Không tự suy diễn giá, rating hoặc URL.
    for key in [
        "product_id",
        "title",
        "name",
        "source",
        "price",
        "currency",
        "rating",
        "rating_count",
        "review_count",
        "product_url",
    ]:
        if key in product:
            compact[key] = product[key]

    for key, limit in [
        ("description", 350),
        ("features", 450),
        ("category", 200),
        ("category_path", 200),
    ]:
        if key in product:
            compact[key] = shorten_text(
                product[key],
                max_chars=limit
            )

    # Hỗ trợ các tên trường review có thể có
    # trong rag_context hiện tại.
    review_key = next(
        (
            key
            for key in [
                "reviews",
                "review_evidence",
                "selected_reviews",
            ]
            if isinstance(product.get(key), list)
        ),
        None
    )

    compact_reviews = []

    if review_key is not None:

        for review in product[review_key]:

            if not isinstance(review, dict):
                continue

            compact_review = {}

            for key in [
                "review_id",
                "rating",
                "review_rating",
                "capacity_check",
                "capacity_status",
            ]:
                if key in review:
                    compact_review[key] = review[key]

            review_text = next(
                (
                    review[key]
                    for key in [
                        "review_text",
                        "rag_review_text",
                        "text",
                        "smart_review_text",
                    ]
                    if review.get(key)
                ),
                ""
            )

            compact_review["review_text"] = (
                shorten_text(review_text, 500)
            )

            compact_reviews.append(compact_review)

    compact["reviews"] = compact_reviews

    return compact


llm_context = {
    "query": rag_context.get(
        "query",
        globals().get(
            "test_query",
            "Tư vấn thẻ nhớ microSD 32GB dưới 15 USD"
        )
    ),
    "products": [
        compact_product(product)
        for product in rag_context["products"]
    ]
}

print("PASS: LLM CONTEXT BUILT")
print("PRODUCTS:", len(llm_context["products"]))

for i, product in enumerate(
    llm_context["products"],
    start=1
):
    print(
        f"{i}.",
        product.get("product_id"),
        "| reviews:",
        len(product["reviews"])
    )

print("\nFIRST PRODUCT KEYS:")
print(list(llm_context["products"][0].keys()))

PASS: LLM CONTEXT BUILT
PRODUCTS: 5
1. amazon_B073JWXGNT | reviews: 6
2. amazon_B011BRUOMO | reviews: 6
3. amazon_B0C2VMSBGX | reviews: 1
4. amazon_B016A5NH6C | reviews: 3
5. amazon_B0BVVJM2X8 | reviews: 5

FIRST PRODUCT KEYS:
['product_id', 'source', 'price', 'currency', 'review_count', 'product_url', 'description', 'reviews']


In [36]:
# CELL 27 — GROUNDED RAG SYSTEM PROMPT

SYSTEM_PROMPT = """
Bạn là chatbot tư vấn sản phẩm thương mại điện tử
sử dụng Retrieval-Augmented Generation (RAG).

Nhiệm vụ:
Trả lời câu hỏi của người dùng dựa trên dữ liệu sản phẩm
và bằng chứng đánh giá được cung cấp.

QUY TẮC BẮT BUỘC:

1. Chỉ sử dụng thông tin có trong RAG CONTEXT.
   Không tự tạo sản phẩm, thông số, giá, đánh giá
   hoặc đường dẫn mua hàng.

2. Giá trong dữ liệu có thể là giá lịch sử.
   Không khẳng định đó là giá bán hiện tại.

3. Giữ nguyên đơn vị tiền tệ trong dữ liệu.
   Không tự quy đổi USD sang VND hoặc ngược lại.

4. Nếu product_url thiếu hoặc null:
   Nói rõ dữ liệu chưa có đường dẫn mua hàng.
   Không tự tạo link Amazon, Shopee hoặc Tiki.

5. Khi nhận xét dựa trên review:
   Dẫn review_id tương ứng, ví dụ:
   [review_0001234].

6. Không được trích dẫn review_id không có
   trong RAG CONTEXT.

7. Phân biệt:
   - Thông số sản phẩm từ metadata.
   - Trải nghiệm người mua từ review.

8. Không khẳng định một review đã xác minh
   khả năng tương thích với thiết bị nếu review
   không cung cấp bằng chứng rõ ràng.

9. Nếu sản phẩm chỉ có ít review:
   Nêu rõ bằng chứng đánh giá còn hạn chế.

10. Nếu thiếu thông tin để trả lời:
    Nói rõ dữ liệu chưa đủ.
    Không suy đoán.

11. Không khẳng định sản phẩm tốt nhất tuyệt đối.
    Chỉ so sánh dựa trên dữ liệu được cung cấp.

12. Trả lời bằng tiếng Việt, rõ ràng, dễ đọc.

CẤU TRÚC TRẢ LỜI:

- Tóm tắt nhu cầu người dùng.
- Đề xuất các sản phẩm phù hợp trong dữ liệu.
- Nêu giá, rating và bằng chứng review khi có.
- So sánh các điểm khác biệt được dữ liệu hỗ trợ.
- Nêu hạn chế của dữ liệu và lưu ý trước khi mua.

Không cần đưa đủ 5 sản phẩm nếu một số sản phẩm
không có thông tin phù hợp hoặc bằng chứng cần thiết.
""".strip()

print("PASS: SYSTEM PROMPT READY")
print("CHARACTERS:", len(SYSTEM_PROMPT))

PASS: SYSTEM PROMPT READY
CHARACTERS: 1689


In [37]:
# CELL 28 — BUILD CHAT PROMPT

import json

USER_QUERY = llm_context["query"]

context_json = json.dumps(
    llm_context,
    ensure_ascii=False,
    indent=2,
    default=str
)

USER_PROMPT = f"""
CÂU HỎI NGƯỜI DÙNG:
{USER_QUERY}

RAG CONTEXT:
{context_json}

Hãy trả lời câu hỏi dựa trên RAG CONTEXT.
""".strip()

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": USER_PROMPT
    }
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

prompt_tokens = model_inputs.shape[-1]

model_context_limit = getattr(
    llm.config,
    "max_position_embeddings",
    None
)

MAX_NEW_TOKENS = 1200

print("PASS: PROMPT BUILT")
print("USER QUERY:", USER_QUERY)
print("PROMPT TOKENS:", prompt_tokens)
print("MODEL CONTEXT LIMIT:", model_context_limit)
print("MAX NEW TOKENS:", MAX_NEW_TOKENS)

if model_context_limit is not None:

    if prompt_tokens + MAX_NEW_TOKENS > model_context_limit:

        print(
            "\nWARNING: Prompt + output vượt "
            "giới hạn context."
        )

    else:

        print(
            "\nPASS: Prompt nằm trong "
            "giới hạn context."
        )

PASS: PROMPT BUILT
USER QUERY: Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video và sử dụng cho điện thoại
PROMPT TOKENS: 2946
MODEL CONTEXT LIMIT: 32768
MAX NEW TOKENS: 1200

PASS: Prompt nằm trong giới hạn context.


In [40]:
# CELL 31 — INSPECT LLM EVIDENCE

print("=" * 70)
print("LLM CONTEXT INSPECTION")
print("=" * 70)

for i, product in enumerate(
    llm_context["products"],
    start=1
):

    print(f"\nPRODUCT {i}")

    print(
        "ID:",
        product.get("product_id")
    )

    print(
        "TITLE:",
        product.get(
            "title",
            product.get("name")
        )
    )

    print(
        "PRICE:",
        product.get("price"),
        product.get("currency")
    )

    print(
        "RATING:",
        product.get("rating")
    )

    print(
        "RATING COUNT:",
        product.get("rating_count")
    )

    print(
        "REVIEW COUNT:",
        product.get("review_count")
    )

    print(
        "EVIDENCE REVIEWS:",
        len(product.get("reviews", []))
    )

    print(
        "REVIEW IDS:",
        [
            r.get("review_id")
            for r in product.get(
                "reviews",
                []
            )
        ]
    )

LLM CONTEXT INSPECTION

PRODUCT 1
ID: amazon_B073JWXGNT
TITLE: None
PRICE: 8.66 USD
RATING: None
RATING COUNT: None
REVIEW COUNT: 233718
EVIDENCE REVIEWS: 6
REVIEW IDS: ['review_0529082', 'review_0426199', 'review_0561905', 'review_0639235', 'review_0198424', 'review_0046578']

PRODUCT 2
ID: amazon_B011BRUOMO
TITLE: None
PRICE: 8.99 USD
RATING: None
RATING COUNT: None
REVIEW COUNT: 115781
EVIDENCE REVIEWS: 6
REVIEW IDS: ['review_0139542', 'review_0070564', 'review_0389996', 'review_0037177', 'review_0025295', 'review_0083613']

PRODUCT 3
ID: amazon_B0C2VMSBGX
TITLE: None
PRICE: 7.99 USD
RATING: None
RATING COUNT: None
REVIEW COUNT: 121
EVIDENCE REVIEWS: 1
REVIEW IDS: ['review_0489045']

PRODUCT 4
ID: amazon_B016A5NH6C
TITLE: None
PRICE: 8.98 USD
RATING: None
RATING COUNT: None
REVIEW COUNT: 16578
EVIDENCE REVIEWS: 3
REVIEW IDS: ['review_0119868', 'review_0206315', 'review_0621556']

PRODUCT 5
ID: amazon_B0BVVJM2X8
TITLE: None
PRICE: 14.49 USD
RATING: None
RATING COUNT: None
REVIEW COUN

In [41]:
# CELL 32 — STRICT GROUNDED RAG PROMPT

SYSTEM_PROMPT = """
Bạn là chatbot tư vấn sản phẩm thương mại điện tử
sử dụng Retrieval-Augmented Generation (RAG).

Chỉ được sử dụng thông tin trong RAG CONTEXT.

QUY TẮC BẮT BUỘC:

1. Không tạo thêm sản phẩm, giá, thông số,
   số lượng đánh giá hoặc đường dẫn mua hàng.

2. Giá trong dữ liệu là giá được ghi nhận
   trong bộ dữ liệu, không phải giá hiện tại.

3. Giữ nguyên đơn vị tiền tệ.
   Không tự quy đổi USD sang VND.

4. Không suy diễn khả năng tương thích
   với một mẫu điện thoại cụ thể.

5. Nếu người dùng chưa cung cấp mẫu điện thoại,
   nhắc họ kiểm tra thiết bị có hỗ trợ microSD
   và dung lượng 32GB hay không.

6. Khi dùng nội dung từ review,
   PHẢI ghi review_id ngay sau nhận xét.

   Ví dụ:
   Một người mua nhận xét thẻ hoạt động tốt
   khi lưu ảnh [review_0001234].

7. Không được trích dẫn review_id không tồn tại
   trong RAG CONTEXT.

8. Nếu không có review hỗ trợ nhận xét,
   không được trình bày nhận xét đó như
   trải nghiệm thực tế của người mua.

9. Phân biệt rõ:
   - rating_count: số lượt đánh giá từ metadata;
   - reviews trong RAG CONTEXT:
     tập review được chọn làm bằng chứng.

10. Không khẳng định đã đọc toàn bộ review.

11. Không sử dụng các kết luận tuyệt đối như:
    "tốt nhất", "chắc chắn tương thích",
    "tất cả đều chất lượng cao".

12. Nếu thiếu product_url:
    Nói rằng dữ liệu chưa có đường dẫn mua hàng.
    Không tự tạo URL.

13. Với sản phẩm có ít review được liên kết,
    nêu rằng bằng chứng review còn hạn chế.

14. Không biến trải nghiệm trên một thiết bị
    thành kết luận cho mọi điện thoại.

15. Chỉ đề xuất sản phẩm đáp ứng điều kiện
    dung lượng và ngân sách trong dữ liệu.

TRẢ LỜI BẰNG TIẾNG VIỆT.

CẤU TRÚC:

1. Tóm tắt nhu cầu.
2. Danh sách sản phẩm phù hợp:
   - Tên sản phẩm
   - Giá ghi nhận trong dữ liệu
   - Rating nếu có
   - Nhận xét có review_id nếu có bằng chứng
3. So sánh ngắn dựa trên dữ liệu.
4. Lưu ý:
   - Giá không phải giá thời gian thực
   - Khả năng tương thích cần kiểm tra
   - Đường dẫn mua hàng nếu thiếu

Không cần đưa đủ 5 sản phẩm.
Không trích nguyên văn review dài bằng tiếng Anh;
hãy tóm tắt bằng tiếng Việt và dẫn review_id.
""".strip()

print("PASS: STRICT SYSTEM PROMPT READY")

PASS: STRICT SYSTEM PROMPT READY


In [42]:
# CELL 33 — GENERATE GROUNDED ANSWER V2

import json
import time
import torch

USER_QUERY = llm_context["query"]

context_json = json.dumps(
    llm_context,
    ensure_ascii=False,
    indent=2,
    default=str
)

USER_PROMPT = f"""
CÂU HỎI:
{USER_QUERY}

RAG CONTEXT:
{context_json}

Hãy tư vấn dựa trên dữ liệu được cung cấp.

Đặc biệt:
- Mỗi nhận xét từ review phải có review_id.
- Không tự tạo URL.
- Không coi giá dữ liệu là giá hiện tại.
- Không khẳng định tương thích với mọi điện thoại.
""".strip()

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": USER_PROMPT
    }
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

prompt_tokens = model_inputs.shape[-1]

MAX_NEW_TOKENS = 1200

print("PROMPT TOKENS:", prompt_tokens)

assert (
    prompt_tokens + MAX_NEW_TOKENS
    <= llm.config.max_position_embeddings
), "Prompt vượt giới hạn context."

start_time = time.time()

with torch.inference_mode():

    outputs = llm.generate(
        input_ids=model_inputs.to(
            llm.device
        ),
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[
    0,
    model_inputs.shape[-1]:
]

chatbot_answer_v2 = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

elapsed_v2 = (
    time.time() - start_time
)

print("\n" + "=" * 70)
print("CHATBOT ANSWER V2")
print("=" * 70)

print(chatbot_answer_v2)

print("\nTIME:", round(elapsed_v2, 2), "seconds")

PROMPT TOKENS: 3108

CHATBOT ANSWER V2
### Câu 1: Nhu cầu của bạn

Bạn cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video và sử dụng cho điện thoại.

### Danh sách sản phẩm phù hợp:

1. **SanDisk Ultra 32GB MicroSDHC UHS-I Card**
   - **Giá**: 8.66 USD
   - **Rating**: Có 5 sao từ 5 đánh giá
   - **Nhận xét**: 
     - [review_0529082]: "I Bought this 32GB Micro SD Card because of its transfer speeds of up to 98MB/s**. Works great in my 12MP DSLR Camera. I have used SanDisk for years and have never had a problem. I highly recommend this product"

2. **SanDisk Ultra 32GB MicroSDHC UHS-I Card**
   - **Giá**: 8.99 USD
   - **Rating**: Có 5 sao từ 5 đánh giá
   - **Nhận xét**: 
     - [review_0139542]: "Purchased for camera use. Excellent memory/storage card and adapter. No negative issues. Affordable price point. Well done. Five stars."

3. **ADATA Premier Micro SDXC/SDHC UHS-1 Class 10 micro SD card**
   - **Giá**: 8.98 USD
   - **Rating**: Có 5 sao từ 5 đánh giá
   - **Nhận xét**: 
     -

In [43]:
# CELL 34 — REVIEW CITATION AUDIT

import re

valid_review_ids = set()

for product in llm_context["products"]:

    for review in product.get(
        "reviews",
        []
    ):

        review_id = review.get(
            "review_id"
        )

        if review_id:

            valid_review_ids.add(
                str(review_id)
            )

cited_review_ids = set(
    re.findall(
        r"review_\d+",
        chatbot_answer_v2
    )
)

invalid_review_ids = (
    cited_review_ids
    - valid_review_ids
)

print("AVAILABLE REVIEW IDS:", len(valid_review_ids))

print("CITED REVIEW IDS:", len(cited_review_ids))

print("INVALID REVIEW IDS:", invalid_review_ids)

if invalid_review_ids:

    print(
        "\nFAIL: Chatbot đã trích dẫn "
        "review_id không có trong context."
    )

elif cited_review_ids:

    print(
        "\nPASS: Các review_id được trích dẫn "
        "đều tồn tại trong RAG Context."
    )

else:

    print(
        "\nWARNING: Chatbot chưa trích dẫn "
        "review_id nào."
    )

AVAILABLE REVIEW IDS: 21
CITED REVIEW IDS: 3
INVALID REVIEW IDS: set()

PASS: Các review_id được trích dẫn đều tồn tại trong RAG Context.


In [38]:
# CELL 29 — GENERATE FIRST RAG CHATBOT ANSWER

import torch
import time

start_time = time.time()

inputs = model_inputs.to(llm.device)

with torch.inference_mode():

    outputs = llm.generate(
        input_ids=inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[
    0,
    inputs.shape[-1]:
]

chatbot_answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

elapsed = time.time() - start_time

print("=" * 70)
print("CHATBOT ANSWER")
print("=" * 70)

print(chatbot_answer)

print("\n" + "=" * 70)
print("GENERATION INFO")
print("=" * 70)

print("MODEL:", LLM_NAME)
print("PROMPT TOKENS:", prompt_tokens)
print("GENERATED TOKENS:", len(generated_tokens))
print("TIME:", round(elapsed, 2), "seconds")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos toke

CHATBOT ANSWER
Theo yêu cầu của bạn, tôi sẽ đề xuất cho bạn một số lựa chọn thẻ nhớ microSD 32GB dưới 15 USD dựa trên dữ liệu có sẵn:

1. Sản phẩm: SanDisk Ultra microSDHC 32GB
Giá: 8.66 USD
Đánh giá: Có 233.718 đánh giá, đa số đều rất tốt với bình luận như sau:
- "I Bought this 32GB Micro SD Card because of its transfer speeds of up to 98MB/s**. Works great in my 12MP DSLR Camera." 
- "Very good memory card" 
- "If you don't want to use your internal cell phone memory to store photos and videos, I recommend this product for you."

2. Sản phẩm: SanDisk Ultra 32GB MicroSDHC UHS-I
Giá: 8.99 USD
Đánh giá: Có 115.781 đánh giá, đa số cũng rất tốt với bình luận như sau:
- "I love the read & write speed of this micro SD card. I will NOT be returning this item even though I can NOT use this item."
- "Nice sized memory card for a good price."
- "Excellent memory card used to store a great deal of photos on my Android phone."

3. Sản phẩm: Adata Premier Micro SDXC/SDHC UHS-1 Class 10 32GB
Giá: 8

In [39]:
# CELL 30 — SAVE FIRST CHATBOT TEST

import json
from pathlib import Path

OUT_DIR = Path("/kaggle/working/step13")
OUT_DIR.mkdir(parents=True, exist_ok=True)

result = {
    "model": LLM_NAME,
    "query": USER_QUERY,
    "product_ids": [
        p.get("product_id")
        for p in llm_context["products"]
    ],
    "prompt_tokens": int(prompt_tokens),
    "generated_tokens": int(
        len(generated_tokens)
    ),
    "generation_seconds": round(
        elapsed,
        2
    ),
    "answer": chatbot_answer
}

output_path = (
    OUT_DIR / "first_rag_chatbot_test.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("PASS: CHATBOT TEST SAVED")
print("FILE:", output_path)

PASS: CHATBOT TEST SAVED
FILE: /kaggle/working/step13/first_rag_chatbot_test.json


In [44]:
# CELL 35 — VERIFY PRODUCT FACTS AGAINST RAG CONTEXT

import json
from pathlib import Path

with open(
    "/kaggle/working/step13/rag_context_test.json",
    "r",
    encoding="utf-8"
) as f:
    original_context = json.load(f)

print("=" * 90)
print("VERIFY PRODUCT FACTS")
print("=" * 90)

for i, product in enumerate(
    original_context["products"],
    start=1
):

    print(f"\nPRODUCT {i}")

    for key, value in product.items():

        if key in {
            "product_id",
            "title",
            "name",
            "price",
            "currency",
            "rating",
            "rating_count",
            "review_count",
            "linked_review_count",
            "product_url"
        }:

            print(
                f"{key}: {value}"
            )

    print(
        "LLM CONTEXT:",
        {
            key: llm_context["products"][i - 1].get(key)
            for key in [
                "rating",
                "rating_count",
                "review_count"
            ]
        }
    )

VERIFY PRODUCT FACTS

PRODUCT 1
product_id: amazon_B073JWXGNT
price: 8.66
currency: USD
review_count: 233718
product_url: None
LLM CONTEXT: {'rating': None, 'rating_count': None, 'review_count': '233718'}

PRODUCT 2
product_id: amazon_B011BRUOMO
price: 8.99
currency: USD
review_count: 115781
product_url: None
LLM CONTEXT: {'rating': None, 'rating_count': None, 'review_count': '115781'}

PRODUCT 3
product_id: amazon_B0C2VMSBGX
price: 7.99
currency: USD
review_count: 121
product_url: None
LLM CONTEXT: {'rating': None, 'rating_count': None, 'review_count': '121'}

PRODUCT 4
product_id: amazon_B016A5NH6C
price: 8.98
currency: USD
review_count: 16578
product_url: None
LLM CONTEXT: {'rating': None, 'rating_count': None, 'review_count': '16578'}

PRODUCT 5
product_id: amazon_B0BVVJM2X8
price: 14.49
currency: USD
review_count: 3241
product_url: None
LLM CONTEXT: {'rating': None, 'rating_count': None, 'review_count': '3241'}


In [45]:
# CELL 36 — VERIFY CITED REVIEW CONTENT

import re

cited_ids = set(
    re.findall(
        r"review_\d+",
        chatbot_answer_v2
    )
)

for product in llm_context["products"]:

    product_id = product.get(
        "product_id"
    )

    for review in product.get(
        "reviews",
        []
    ):

        if review.get("review_id") in cited_ids:

            print("=" * 90)

            print(
                "PRODUCT:",
                product_id
            )

            print(
                "REVIEW ID:",
                review.get("review_id")
            )

            print(
                "REVIEW RATING:",
                review.get(
                    "rating",
                    review.get("review_rating")
                )
            )

            print(
                "REVIEW TEXT:",
                review.get("review_text")
            )

PRODUCT: amazon_B073JWXGNT
REVIEW ID: review_0529082
REVIEW RATING: 5.0
REVIEW TEXT: I Bought this 32GB Micro SD Card because of its transfer speeds of up to 98MB/s**, Works great in my 12MP DSLR Camera. I have used SanDisk for years and have never had a problem. I highly recommend this product
PRODUCT: amazon_B011BRUOMO
REVIEW ID: review_0139542
REVIEW RATING: 5.0
REVIEW TEXT: I love the read & write speed of this micro SD card. I will NOT be returning this item even though I can NOT use this item. I purchased this card for my cell phone, apparently I have an older cell phone which does not support 64 gigs of space. So I recommend you google your cellphone and see how much space it actually supports. My LG Optimus phone will only support micro SD cards up to 32 gigabytes. So I have just ordered the $15 32gb SanDisk Micro SD Card because that is cheaper than a $120...
PRODUCT: amazon_B016A5NH6C
REVIEW ID: review_0119868
REVIEW RATING: 5.0
REVIEW TEXT: I needed a micro card and this was

In [46]:
# CELL 37 — INSPECT PRODUCT RATING COLUMNS

print("PRODUCTS SHAPE:", products.shape)

candidate_columns = [
    col
    for col in products.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "rating",
            "review",
            "price",
            "currency"
        ]
    )
]

print("\nCANDIDATE COLUMNS:")

for col in candidate_columns:
    print("-", col)

test_product_ids = [
    p["product_id"]
    for p in rag_context["products"]
]

display(
    products.loc[
        products["product_id"].isin(
            test_product_ids
        ),
        [
            "product_id"
        ] + candidate_columns
    ]
)

PRODUCTS SHAPE: (164926, 60)

CANDIDATE COLUMNS:
- price
- currency
- average_rating
- review_count
- smart_currency
- smart_price
- smart_rating
- smart_review_count
- has_valid_price
- has_valid_currency
- price_filter_eligible
- etl_review_rows
- etl_review_text_rows
- etl_review_evidence_rows
- has_review_evidence


,product_id,price,currency,average_rating,review_count,smart_currency,smart_price,smart_rating,smart_review_count,has_valid_price,has_valid_currency,price_filter_eligible,etl_review_rows,etl_review_text_rows,etl_review_evidence_rows,has_review_evidence
13,amazon_B073JWXGNT,8.66,USD,4.7,233718,USD,8.66,4.7,233718.0,True,True,True,302,302,302,True
60,amazon_B011BRUOMO,8.99,USD,4.6,115781,USD,8.99,4.6,115781.0,True,True,True,1354,1354,1354,True
1138,amazon_B016A5NH6C,8.98,USD,4.4,16578,USD,8.98,4.4,16578.0,True,True,True,4,4,4,True
8034,amazon_B0BVVJM2X8,14.49,USD,4.6,3241,USD,14.49,4.6,3241.0,True,True,True,5,5,5,True
77691,amazon_B0C2VMSBGX,7.99,USD,4.5,121,USD,7.99,4.5,121.0,True,True,True,1,1,1,True


In [47]:
# CELL 38 — BUILD CORRECTED LLM CONTEXT V3

import json
import pandas as pd
from pathlib import Path

# Lấy metadata trực tiếp từ bảng products
product_lookup = (
    products
    .drop_duplicates("product_id")
    .set_index("product_id")
)

def safe_value(value):
    """Chuyển NaN/NaT thành None để JSON hợp lệ."""
    if value is None:
        return None

    if isinstance(value, (list, dict)):
        return value

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if hasattr(value, "item"):
        try:
            return value.item()
        except (AttributeError, ValueError):
            pass

    return value


def shorten_text_v3(value, max_chars=500):
    value = safe_value(value)

    if value is None:
        return ""

    text = str(value).strip()

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rsplit(" ", 1)[0] + "..."


llm_products_v3 = []

for rag_product in rag_context["products"]:

    product_id = rag_product["product_id"]

    assert product_id in product_lookup.index, (
        f"Không tìm thấy sản phẩm: {product_id}"
    )

    metadata = product_lookup.loc[product_id]

    # Giữ các review đã truy xuất từ RAG Context
    original_reviews = next(
        (
            rag_product[key]
            for key in [
                "reviews",
                "review_evidence",
                "selected_reviews"
            ]
            if isinstance(rag_product.get(key), list)
        ),
        []
    )

    compact_reviews = []

    for review in original_reviews:

        review_text = next(
            (
                review[key]
                for key in [
                    "review_text",
                    "rag_review_text",
                    "text",
                    "smart_review_text"
                ]
                if review.get(key)
            ),
            ""
        )

        compact_reviews.append({
            "review_id": safe_value(
                review.get("review_id")
            ),
            "review_rating": safe_value(
                review.get(
                    "rating",
                    review.get("review_rating")
                )
            ),
            "review_text": shorten_text_v3(
                review_text,
                500
            )
        })

    product_v3 = {
        "product_id": product_id,

        "title": safe_value(
            metadata.get("title")
        ),

        "source": safe_value(
            metadata.get("source")
        ),

        "price": safe_value(
            metadata.get("smart_price")
        ),

        "currency": safe_value(
            metadata.get("smart_currency")
        ),

        # Rating trung bình của sản phẩm
        "product_average_rating": safe_value(
            metadata.get("smart_rating")
        ),

        # Số lượt đánh giá từ metadata
        "metadata_rating_count": safe_value(
            metadata.get("smart_review_count")
        ),

        # Số review có trong dataset
        "linked_review_count": safe_value(
            metadata.get("etl_review_rows")
        ),

        # Số review thực sự đưa vào prompt
        "retrieved_evidence_count": len(
            compact_reviews
        ),

        "product_url": safe_value(
            metadata.get("product_url")
        ),

        "description": shorten_text_v3(
            rag_product.get(
                "description",
                metadata.get("description")
            ),
            350
        ),

        "features": shorten_text_v3(
            rag_product.get(
                "features",
                metadata.get("features")
            ),
            450
        ),

        "reviews": compact_reviews
    }

    llm_products_v3.append(
        product_v3
    )


llm_context_v3 = {
    "query": llm_context["query"],
    "products": llm_products_v3
}

print("PASS: LLM CONTEXT V3 BUILT")
print("PRODUCTS:", len(llm_products_v3))

for i, p in enumerate(
    llm_products_v3,
    start=1
):

    print(f"\n{i}. {p['product_id']}")

    print(
        "Price:",
        p["price"],
        p["currency"]
    )

    print(
        "Average rating:",
        p["product_average_rating"]
    )

    print(
        "Metadata rating count:",
        p["metadata_rating_count"]
    )

    print(
        "Linked reviews:",
        p["linked_review_count"]
    )

    print(
        "Retrieved evidence:",
        p["retrieved_evidence_count"]
    )

PASS: LLM CONTEXT V3 BUILT
PRODUCTS: 5

1. amazon_B073JWXGNT
Price: 8.66 USD
Average rating: 4.7
Metadata rating count: 233718.0
Linked reviews: 302
Retrieved evidence: 6

2. amazon_B011BRUOMO
Price: 8.99 USD
Average rating: 4.6
Metadata rating count: 115781.0
Linked reviews: 1354
Retrieved evidence: 6

3. amazon_B0C2VMSBGX
Price: 7.99 USD
Average rating: 4.5
Metadata rating count: 121.0
Linked reviews: 1
Retrieved evidence: 1

4. amazon_B016A5NH6C
Price: 8.98 USD
Average rating: 4.4
Metadata rating count: 16578.0
Linked reviews: 4
Retrieved evidence: 3

5. amazon_B0BVVJM2X8
Price: 14.49 USD
Average rating: 4.6
Metadata rating count: 3241.0
Linked reviews: 5
Retrieved evidence: 5


In [49]:
# CELL 39 — VALIDATE LLM CONTEXT V3

assert len(llm_context_v3["products"]) == 5

for p in llm_context_v3["products"]:

    assert p["product_average_rating"] is not None, (
        f"Thiếu rating: {p['product_id']}"
    )

    assert p["metadata_rating_count"] is not None, (
        f"Thiếu rating count: {p['product_id']}"
    )

    assert (
        p["retrieved_evidence_count"]
        == len(p["reviews"])
    )

    assert p["price"] is not None

    assert p["currency"] == "USD"

print("PASS: ALL PRODUCT FACTS VALIDATED")

OUT_DIR = Path("/kaggle/working/step13")
OUT_DIR.mkdir(parents=True, exist_ok=True)

context_v3_path = (
    OUT_DIR / "llm_context_v3.json"
)

with open(
    context_v3_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        llm_context_v3,
        f,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    )

print("SAVED:", context_v3_path)

PASS: ALL PRODUCT FACTS VALIDATED
SAVED: /kaggle/working/step13/llm_context_v3.json


In [50]:
# CELL 40 — BUILD PROMPT V3

import json

USER_QUERY = llm_context_v3["query"]

context_json_v3 = json.dumps(
    llm_context_v3,
    ensure_ascii=False,
    indent=2,
    allow_nan=False
)

USER_PROMPT_V3 = f"""
CÂU HỎI NGƯỜI DÙNG:
{USER_QUERY}

RAG CONTEXT:
{context_json_v3}

HƯỚNG DẪN DIỄN GIẢI DỮ LIỆU:

- product_average_rating:
  Điểm đánh giá trung bình của sản phẩm trên thang 5.

- metadata_rating_count:
  Số lượt đánh giá ghi nhận trong metadata.
  KHÔNG phải số review văn bản đã truy xuất.

- linked_review_count:
  Số review liên kết với sản phẩm trong dataset.

- retrieved_evidence_count:
  Số review được chọn đưa vào RAG Context.

- review_rating:
  Điểm của một review cá nhân.
  KHÔNG phải điểm trung bình của sản phẩm.

YÊU CẦU TRẢ LỜI:

1. Ghi chính xác rating trung bình của từng sản phẩm.
2. Không nhầm review_rating với product_average_rating.
3. Chỉ tóm tắt review bằng tiếng Việt, không chép
   nguyên văn đoạn review tiếng Anh.
4. Mỗi nhận xét dựa trên review phải có review_id.
5. Không khẳng định sản phẩm tương thích với điện thoại
   cụ thể nếu chưa biết mẫu điện thoại.
6. Giá là giá ghi nhận trong dataset, không phải
   giá bán thời gian thực.
7. Không tự tạo đường dẫn mua hàng.
8. Không suy diễn tốc độ ghi, độ bền hoặc chất lượng
   từ những thông tin không có trong context.
9. Không khẳng định đã đọc toàn bộ đánh giá.
""".strip()

messages_v3 = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": USER_PROMPT_V3
    }
]

model_inputs_v3 = tokenizer.apply_chat_template(
    messages_v3,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

prompt_tokens_v3 = model_inputs_v3.shape[-1]

MAX_NEW_TOKENS_V3 = 1200

print("PASS: PROMPT V3 BUILT")
print("PROMPT TOKENS:", prompt_tokens_v3)
print(
    "MODEL CONTEXT LIMIT:",
    llm.config.max_position_embeddings
)

assert (
    prompt_tokens_v3 + MAX_NEW_TOKENS_V3
    <= llm.config.max_position_embeddings
), "Prompt vượt giới hạn context."

PASS: PROMPT V3 BUILT
PROMPT TOKENS: 3403
MODEL CONTEXT LIMIT: 32768


In [51]:
# CELL 41 — GENERATE CHATBOT ANSWER V3

import torch
import time

start_time_v3 = time.time()

with torch.inference_mode():

    outputs_v3 = llm.generate(
        input_ids=model_inputs_v3.to(
            llm.device
        ),
        max_new_tokens=MAX_NEW_TOKENS_V3,
        do_sample=False,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens_v3 = outputs_v3[
    0,
    model_inputs_v3.shape[-1]:
]

chatbot_answer_v3 = tokenizer.decode(
    generated_tokens_v3,
    skip_special_tokens=True
).strip()

elapsed_v3 = (
    time.time() - start_time_v3
)

print("=" * 70)
print("CHATBOT ANSWER V3")
print("=" * 70)

print(chatbot_answer_v3)

print("\n" + "=" * 70)
print("GENERATION INFO")
print("=" * 70)

print("PROMPT TOKENS:", prompt_tokens_v3)

print(
    "GENERATED TOKENS:",
    len(generated_tokens_v3)
)

print(
    "TIME:",
    round(elapsed_v3, 2),
    "seconds"
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


CHATBOT ANSWER V3
### 1. Tóm tắt nhu cầu
Người dùng cần thẻ nhớ microSD 32GB dưới 15 USD để lưu trữ video và sử dụng cho điện thoại.

### 2. Danh sách sản phẩm phù hợp:
1. **Tên sản phẩm:** SanDisk Ultra microSDXC và microSDHC
   - **Giá:** 8.66 USD
   - **Rating:** 4.7
   - **Nhận xét:** 
     - "[review_0529082]": "Card hoạt động tốt với tốc độ truyền tải lên đến 98MB/s, phù hợp cho máy ảnh DSLR của tôi. Tôi đã sử dụng SanDisk trong nhiều năm và chưa bao giờ gặp lỗi. Tôi khuyên bạn nên mua sản phẩm này."

2. **Tên sản phẩm:** SanDisk Ultra 32GB microSDHC UHS-I
   - **Giá:** 8.99 USD
   - **Rating:** 4.6
   - **Nhận xét:** 
     - "[review_0139542]": "Card hoạt động tốt và không gặp bất kỳ vấn đề nào. Tốc độ đọc và ghi ổn định. Giá rất hợp lý."

3. **Tên sản phẩm:** SanDisk Memory Zone 32GB microSDHC UHS-I
   - **Giá:** 7.99 USD
   - **Rating:** 4.5
   - **Nhận xét:** 
     - "[review_0489045]": "Card hoạt động tốt và không gặp bất kỳ vấn đề nào. Tốc độ đọc và ghi ổn định. Tuy nhiên, 

In [52]:
# CELL 42 — AUTOMATIC ANSWER AUDIT V3

import re

valid_review_ids_v3 = {
    str(review["review_id"])
    for product in llm_context_v3["products"]
    for review in product["reviews"]
    if review.get("review_id")
}

cited_review_ids_v3 = set(
    re.findall(
        r"review_\d+",
        chatbot_answer_v3
    )
)

invalid_review_ids_v3 = (
    cited_review_ids_v3
    - valid_review_ids_v3
)

print("=" * 70)
print("ANSWER AUDIT V3")
print("=" * 70)

print(
    "AVAILABLE REVIEW IDS:",
    len(valid_review_ids_v3)
)

print(
    "CITED REVIEW IDS:",
    len(cited_review_ids_v3)
)

print(
    "INVALID REVIEW IDS:",
    invalid_review_ids_v3
)

print("\nPRODUCT FACTS:")

for product in llm_context_v3["products"]:

    print(
        product["product_id"],
        "| rating:",
        product["product_average_rating"],
        "| price:",
        product["price"],
        product["currency"],
        "| metadata ratings:",
        product["metadata_rating_count"],
        "| retrieved reviews:",
        product["retrieved_evidence_count"]
    )

if invalid_review_ids_v3:

    print(
        "\nFAIL: Có review_id không tồn tại "
        "trong RAG Context."
    )

elif cited_review_ids_v3:

    print(
        "\nPASS: Các review_id được trích dẫn "
        "đều tồn tại trong RAG Context."
    )

else:

    print(
        "\nWARNING: Câu trả lời chưa có "
        "review_id."
    )

ANSWER AUDIT V3
AVAILABLE REVIEW IDS: 21
CITED REVIEW IDS: 3
INVALID REVIEW IDS: set()

PRODUCT FACTS:
amazon_B073JWXGNT | rating: 4.7 | price: 8.66 USD | metadata ratings: 233718.0 | retrieved reviews: 6
amazon_B011BRUOMO | rating: 4.6 | price: 8.99 USD | metadata ratings: 115781.0 | retrieved reviews: 6
amazon_B0C2VMSBGX | rating: 4.5 | price: 7.99 USD | metadata ratings: 121.0 | retrieved reviews: 1
amazon_B016A5NH6C | rating: 4.4 | price: 8.98 USD | metadata ratings: 16578.0 | retrieved reviews: 3
amazon_B0BVVJM2X8 | rating: 4.6 | price: 14.49 USD | metadata ratings: 3241.0 | retrieved reviews: 5

PASS: Các review_id được trích dẫn đều tồn tại trong RAG Context.


In [53]:
# CELL 43 — REVIEW CAPACITY CONFLICT AUDIT

import re

EXPECTED_CAPACITY_GB = 32

def extract_capacity_mentions(text):
    """
    Trích xuất các dung lượng GB/TB
    xuất hiện trong nội dung review.
    """

    if not isinstance(text, str):
        return []

    pattern = (
        r"(?<!\w)"
        r"(\d+(?:\.\d+)?)"
        r"\s*"
        r"(GB|TB)"
        r"\b"
    )

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    capacities = []

    for value, unit in matches:

        capacity = float(value)

        if unit.upper() == "TB":
            capacity *= 1024

        capacities.append(capacity)

    return capacities


def audit_review_capacity(
    review_text,
    expected_gb=32
):

    mentions = extract_capacity_mentions(
        review_text
    )

    other_capacities = [
        value
        for value in mentions
        if value != expected_gb
    ]

    if other_capacities:

        return {
            "status": "CAPACITY_AMBIGUOUS",
            "mentions_gb": mentions
        }

    if mentions:

        return {
            "status": "CAPACITY_MATCH",
            "mentions_gb": mentions
        }

    return {
        "status": "CAPACITY_UNSPECIFIED",
        "mentions_gb": []
    }


print("=" * 90)
print("REVIEW CAPACITY AUDIT")
print("=" * 90)

for product in llm_context_v3["products"]:

    print(
        "\nPRODUCT:",
        product["product_id"]
    )

    for review in product["reviews"]:

        result = audit_review_capacity(
            review["review_text"],
            EXPECTED_CAPACITY_GB
        )

        print(
            review["review_id"],
            "|",
            result["status"],
            "|",
            result["mentions_gb"]
        )

        if result["status"] == "CAPACITY_AMBIGUOUS":

            print(
                "TEXT:",
                review["review_text"]
            )

REVIEW CAPACITY AUDIT

PRODUCT: amazon_B073JWXGNT
review_0529082 | CAPACITY_MATCH | [32.0]
review_0426199 | CAPACITY_UNSPECIFIED | []
review_0561905 | CAPACITY_UNSPECIFIED | []
review_0639235 | CAPACITY_UNSPECIFIED | []
review_0198424 | CAPACITY_UNSPECIFIED | []
review_0046578 | CAPACITY_UNSPECIFIED | []

PRODUCT: amazon_B011BRUOMO
review_0139542 | CAPACITY_MATCH | [32.0]
review_0070564 | CAPACITY_UNSPECIFIED | []
review_0389996 | CAPACITY_MATCH | [32.0]
review_0037177 | CAPACITY_UNSPECIFIED | []
review_0025295 | CAPACITY_UNSPECIFIED | []
review_0083613 | CAPACITY_UNSPECIFIED | []

PRODUCT: amazon_B0C2VMSBGX
review_0489045 | CAPACITY_UNSPECIFIED | []

PRODUCT: amazon_B016A5NH6C
review_0119868 | CAPACITY_UNSPECIFIED | []
review_0206315 | CAPACITY_UNSPECIFIED | []
review_0621556 | CAPACITY_UNSPECIFIED | []

PRODUCT: amazon_B0BVVJM2X8
review_0039559 | CAPACITY_UNSPECIFIED | []
review_0605977 | CAPACITY_UNSPECIFIED | []
review_0004126 | CAPACITY_UNSPECIFIED | []
review_0292437 | CAPACITY_U

In [54]:
# CELL 44 — CHECK DYNAMIC RETRIEVAL COMPONENTS

import faiss
import torch
import pandas as pd

print("=" * 70)
print("DYNAMIC RETRIEVAL CHECK")
print("=" * 70)

required_variables = [
    "products",
    "reviews",
    "llm",
    "tokenizer"
]

for name in required_variables:
    print(
        name,
        "PASS" if name in globals() else "MISSING"
    )

print("\nPRODUCTS:", products.shape)
print("REVIEWS:", reviews.shape)

print("\nFAISS / EMBEDDING VARIABLES:")

for name in [
    "index",
    "faiss_index",
    "product_index",
    "embedding_model",
    "review_model",
    "product_metadata",
    "embedding_metadata"
]:
    if name in globals():
        obj = globals()[name]

        print(
            name,
            "| type:",
            type(obj).__name__,
            "| shape:",
            getattr(obj, "shape", None),
            "| ntotal:",
            getattr(obj, "ntotal", None)
        )

print("\nGPU:", torch.cuda.is_available())
print("LLM:", LLM_NAME)

print("\nPASS: CHECK COMPLETED")

DYNAMIC RETRIEVAL CHECK
products PASS
reviews PASS
llm PASS
tokenizer PASS

PRODUCTS: (164926, 60)
REVIEWS: (767179, 16)

FAISS / EMBEDDING VARIABLES:
review_model | type: SentenceTransformer | shape: None | ntotal: None

GPU: True
LLM: Qwen/Qwen2.5-3B-Instruct

PASS: CHECK COMPLETED


In [55]:
# CELL 45 — LOCATE FAISS FILES

from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

patterns = [
    "*product_faiss*.index",
    "*product_embedding_metadata*.parquet",
    "*step11_config*.json",
    "*step12_config*.json",
]

for pattern in patterns:
    print(f"\nSEARCH: {pattern}")

    matches = list(INPUT_DIR.rglob(pattern))

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("NOT FOUND")


SEARCH: *product_faiss*.index
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/product_faiss_moi.index

SEARCH: *product_embedding_metadata*.parquet
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/product_embedding_metadata.parquet

SEARCH: *step11_config*.json
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/step11_config.json

SEARCH: *step12_config*.json
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/step12_config.json


In [56]:
# CELL 46 — LOAD FAISS AND METADATA

import faiss
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

index_files = list(
    INPUT_DIR.rglob("product_faiss_moi.index")
)

if not index_files:
    index_files = list(
        INPUT_DIR.rglob("product_faiss.index")
    )

metadata_files = list(
    INPUT_DIR.rglob(
        "product_embedding_metadata.parquet"
    )
)

assert len(index_files) == 1, (
    f"Cần xác định đúng 1 file FAISS, "
    f"nhưng tìm thấy: {index_files}"
)

assert len(metadata_files) == 1, (
    f"Cần xác định đúng 1 file metadata, "
    f"nhưng tìm thấy: {metadata_files}"
)

FAISS_PATH = index_files[0]
METADATA_PATH = metadata_files[0]

faiss_index = faiss.read_index(
    str(FAISS_PATH)
)

embedding_metadata = pd.read_parquet(
    METADATA_PATH
)

print("PASS: FAISS LOADED")

print("\nFAISS FILE:", FAISS_PATH)
print("METADATA FILE:", METADATA_PATH)

print("\nFAISS VECTORS:", faiss_index.ntotal)
print("FAISS DIMENSION:", faiss_index.d)

print("\nMETADATA SHAPE:", embedding_metadata.shape)

print("\nMETADATA COLUMNS:")
print(embedding_metadata.columns.tolist())

assert faiss_index.ntotal == len(
    embedding_metadata
), "Số vector FAISS không khớp metadata."

assert faiss_index.d == 384, (
    "FAISS dimension không phải 384."
)

print("\nPASS: FAISS SIZE AND DIMENSION VALIDATED")

PASS: FAISS LOADED

FAISS FILE: /kaggle/input/datasets/trnquangtriu/13dataset/product_faiss_moi.index
METADATA FILE: /kaggle/input/datasets/trnquangtriu/13dataset/product_embedding_metadata.parquet

FAISS VECTORS: 164926
FAISS DIMENSION: 384

METADATA SHAPE: (164926, 17)

METADATA COLUMNS:
['faiss_row_id', 'product_id', 'product_name', 'source', 'category', 'subcategory', 'category_path', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'description', 'product_features', 'product_url', 'retrieval_text', 'embedding_text_v2']

PASS: FAISS SIZE AND DIMENSION VALIDATED


In [57]:
# CELL 47 — VALIDATE FAISS ROW MAPPING

assert "faiss_row_id" in embedding_metadata.columns, (
    "Metadata thiếu cột faiss_row_id."
)

assert "product_id" in embedding_metadata.columns, (
    "Metadata thiếu cột product_id."
)

assert embedding_metadata[
    "faiss_row_id"
].is_unique, (
    "faiss_row_id bị trùng."
)

assert embedding_metadata[
    "product_id"
].is_unique, (
    "product_id bị trùng."
)

expected_ids = set(
    range(faiss_index.ntotal)
)

actual_ids = set(
    embedding_metadata[
        "faiss_row_id"
    ].astype(int)
)

assert actual_ids == expected_ids, (
    "faiss_row_id không liên tục "
    "hoặc không khớp FAISS."
)

metadata_by_faiss_row = (
    embedding_metadata
    .set_index("faiss_row_id")
    .sort_index()
)

print("PASS: FAISS ROW MAPPING VALIDATED")

print(
    metadata_by_faiss_row[
        ["product_id"]
    ].head()
)

PASS: FAISS ROW MAPPING VALIDATED
                     product_id
faiss_row_id                   
0             amazon_B07PXGQC1Q
1             amazon_B07KTYJ769
2             amazon_B0BGNG1294
3             amazon_B08CLNX58K
4             amazon_B08WJSHSLC


In [58]:
# CELL 48 — INSPECT STEP 11 CONFIG

import json

config_files = list(
    Path("/kaggle/input").rglob(
        "step11_config.json"
    )
)

assert len(config_files) == 1, (
    f"Tìm thấy {len(config_files)} "
    "file step11_config.json."
)

with open(
    config_files[0],
    "r",
    encoding="utf-8"
) as f:
    step11_config = json.load(f)

print("STEP 11 CONFIG:")
print(
    json.dumps(
        step11_config,
        ensure_ascii=False,
        indent=2
    )
)

print("\nCURRENT REVIEW MODEL:")
print(
    getattr(
        review_model,
        "_model_card_text",
        "Model đang được load"
    )[:300]
)

print("\nMODEL DIMENSION:")
print(
    review_model.get_sentence_embedding_dimension()
)

print("\nMAX SEQUENCE LENGTH:")
print(
    review_model.max_seq_length
)

STEP 11 CONFIG:
{
  "step": "step11_product_embedding_faiss",
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "model_max_sequence_length": 128,
  "embedding_text_column": "embedding_text_v2",
  "normalization": true,
  "faiss_index_type": "IndexFlatIP",
  "similarity": "cosine",
  "total_products": 164926,
  "chunk_size": 4096,
  "batch_size": 64,
  "source_dataset": "step10_unified_products",
  "faiss_mapping_column": "faiss_row_id"
}

CURRENT REVIEW MODEL:
---
language:
- multilingual
- ar
- bg
- ca
- cs
- da
- de
- el
- en
- es
- et
- fa
- fi
- fr
- gl
- gu
- he
- hi
- hr
- hu
- hy
- id
- it
- ja
- ka
- ko
- ku
- lt
- lv
- mk
- mn
- mr
- ms
- my
- nb
- nl
- pl
- pt
- ro
- ru
- sk
- sl
- sq
- sr
- sv
- th
- tr
- uk
- ur
- vi
license: apache-2.0
librar

MODEL DIMENSION:
384

MAX SEQUENCE LENGTH:
128


/tmp/ipykernel_267/729433751.py:43: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  review_model.get_sentence_embedding_dimension()


In [59]:
# CELL 49 — PREPARE PRODUCT QUERY EMBEDDING MODEL

from sentence_transformers import SentenceTransformer
import torch

EXPECTED_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

def get_loaded_model_name(model):

    try:
        return str(
            model._first_module()
            .auto_model
            .config
            ._name_or_path
        )

    except Exception:
        return ""


loaded_model_name = get_loaded_model_name(
    review_model
)

print("CURRENT MODEL:", loaded_model_name)

if (
    "paraphrase-multilingual-MiniLM-L12-v2"
    in loaded_model_name
):

    product_query_model = review_model

    print(
        "Reusing existing MiniLM model."
    )

else:

    product_query_model = (
        SentenceTransformer(
            EXPECTED_MODEL,
            device=(
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
        )
    )

    print(
        "Loaded product query model."
    )


assert (
    product_query_model
    .get_sentence_embedding_dimension()
    == faiss_index.d
)

product_query_model.max_seq_length = 128

print("\nPASS: PRODUCT EMBEDDING MODEL READY")

print(
    "DIMENSION:",
    product_query_model
    .get_sentence_embedding_dimension()
)

print(
    "MAX SEQUENCE LENGTH:",
    product_query_model.max_seq_length
)

CURRENT MODEL: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Reusing existing MiniLM model.

PASS: PRODUCT EMBEDDING MODEL READY
DIMENSION: 384
MAX SEQUENCE LENGTH: 128


/tmp/ipykernel_267/3079379567.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()
/tmp/ipykernel_267/3079379567.py:73: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


In [60]:
# CELL 50 — PARSE USER REQUIREMENTS

import re


def parse_user_requirements(query):

    text = str(query).strip()

    text_lower = text.lower()

    requirements = {
        "original_query": text,
        "capacity_gb": None,
        "max_price": None,
        "currency": None
    }

    # ---------------------------------
    # 1. Dung lượng GB / TB
    # ---------------------------------

    capacity_pattern = (
        r"(?<!\w)"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(GB|TB)"
        r"\b"
    )

    capacity_matches = re.findall(
        capacity_pattern,
        text,
        flags=re.IGNORECASE
    )

    if capacity_matches:

        value, unit = capacity_matches[0]

        capacity = float(
            value.replace(",", ".")
        )

        if unit.upper() == "TB":
            capacity *= 1024

        requirements[
            "capacity_gb"
        ] = capacity

    # ---------------------------------
    # 2. Ngân sách USD
    # ---------------------------------

    usd_pattern = (
        r"(?:dưới|không quá|tối đa|"
        r"chưa đến|under|below|up to)"
        r"\s*"
        r"\$?\s*"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(?:USD|đô|dollars?|\$)"
    )

    usd_match = re.search(
        usd_pattern,
        text,
        flags=re.IGNORECASE
    )

    if usd_match:

        requirements["max_price"] = float(
            usd_match.group(1).replace(
                ",",
                "."
            )
        )

        requirements["currency"] = "USD"

    # ---------------------------------
    # 3. Ngân sách VND
    # ---------------------------------

    vnd_pattern = (
        r"(?:dưới|không quá|tối đa|"
        r"chưa đến|under|below|up to)"
        r"\s*"
        r"(\d+(?:[.,]\d+)?)"
        r"\s*"
        r"(triệu|nghìn|ngàn|"
        r"VND|VNĐ|đồng)"
    )

    vnd_match = re.search(
        vnd_pattern,
        text,
        flags=re.IGNORECASE
    )

    if vnd_match:

        amount = float(
            vnd_match.group(1).replace(
                ",",
                "."
            )
        )

        unit = vnd_match.group(2).lower()

        if unit == "triệu":
            amount *= 1_000_000

        elif unit in {
            "nghìn",
            "ngàn"
        }:
            amount *= 1_000

        requirements["max_price"] = amount

        requirements["currency"] = "VND"

    # ---------------------------------
    # 4. Có nhắc USD nhưng không có
    #    giới hạn ngân sách rõ ràng
    # ---------------------------------

    if (
        requirements["currency"] is None
        and re.search(
            r"\bUSD\b|\$",
            text,
            flags=re.IGNORECASE
        )
    ):

        requirements["currency"] = "USD"

    return requirements


test_query_dynamic = (
    "Tôi cần thẻ nhớ microSD 32GB "
    "dưới 15 USD để lưu video "
    "và sử dụng cho điện thoại"
)

requirements = parse_user_requirements(
    test_query_dynamic
)

print("PASS: REQUIREMENTS PARSED")

print(requirements)

PASS: REQUIREMENTS PARSED
{'original_query': 'Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video và sử dụng cho điện thoại', 'capacity_gb': 32.0, 'max_price': 15.0, 'currency': 'USD'}


In [61]:
# CELL 51 — DYNAMIC FAISS PRODUCT SEARCH

import numpy as np
import pandas as pd
import re


def extract_title_capacities_gb(title):

    if not isinstance(title, str):
        return []

    pattern = (
        r"(?<!\w)"
        r"(\d+(?:\.\d+)?)"
        r"\s*"
        r"(GB|TB)"
        r"\b"
    )

    matches = re.findall(
        pattern,
        title,
        flags=re.IGNORECASE
    )

    capacities = []

    for value, unit in matches:

        capacity = float(value)

        if unit.upper() == "TB":
            capacity *= 1024

        capacities.append(capacity)

    return capacities


def dynamic_semantic_search(
    query,
    top_k=5,
    search_k=1000
):

    requirements = (
        parse_user_requirements(query)
    )

    # ---------------------------------
    # 1. Query embedding
    # ---------------------------------

    query_embedding = (
        product_query_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    assert (
        query_embedding.shape[1]
        == faiss_index.d
    )

    # ---------------------------------
    # 2. FAISS search
    # ---------------------------------

    search_k = min(
        int(search_k),
        faiss_index.ntotal
    )

    similarities, row_ids = (
        faiss_index.search(
            query_embedding,
            search_k
        )
    )

    # ---------------------------------
    # 3. Mapping + filtering
    # ---------------------------------

    candidates = []

    expected_capacity = (
        requirements["capacity_gb"]
    )

    max_price = (
        requirements["max_price"]
    )

    expected_currency = (
        requirements["currency"]
    )

    for similarity, row_id in zip(
        similarities[0],
        row_ids[0]
    ):

        if row_id < 0:
            continue

        metadata = (
            metadata_by_faiss_row.loc[
                int(row_id)
            ]
        )

        product_id = metadata[
            "product_id"
        ]

        title = metadata.get(
            "product_name",
            ""
        )

        # -----------------------------
        # Capacity filter
        # -----------------------------

        if expected_capacity is not None:

            title_capacities = (
                extract_title_capacities_gb(
                    title
                )
            )

            # Lọc bảo thủ:
            # Chỉ nhận sản phẩm có dung lượng
            # được xác định trong tên.
            if expected_capacity not in (
                title_capacities
            ):

                continue

        # -----------------------------
        # Price / currency filter
        # -----------------------------

        price = pd.to_numeric(
            metadata.get("price"),
            errors="coerce"
        )

        currency = metadata.get(
            "currency"
        )

        if expected_currency is not None:

            if (
                str(currency).upper()
                != expected_currency
            ):

                continue

        if max_price is not None:

            if pd.isna(price):
                continue

            if price > max_price:
                continue

        # -----------------------------
        # Rating
        # -----------------------------

        rating = pd.to_numeric(
            metadata.get(
                "average_rating"
            ),
            errors="coerce"
        )

        review_count = pd.to_numeric(
            metadata.get(
                "review_count"
            ),
            errors="coerce"
        )

        rating_score = (
            float(rating) / 5
            if pd.notna(rating)
            else 0
        )

        review_count_value = (
            max(
                0,
                float(review_count)
            )
            if pd.notna(review_count)
            else 0
        )

        review_score = (
            np.log1p(
                review_count_value
            )
            / np.log1p(
                1_000_000
            )
        )

        review_score = float(
            np.clip(
                review_score,
                0,
                1
            )
        )

        # -----------------------------
        # Hybrid metadata ranking
        # Chưa có BPR
        # -----------------------------

        semantic_score = float(
            similarity
        )

        final_score = (
            0.75 * semantic_score
            + 0.20 * rating_score
            + 0.05 * review_score
        )

        candidates.append({

            "product_id":
                product_id,

            "product_name":
                title,

            "source":
                metadata.get(
                    "source"
                ),

            "price":
                (
                    float(price)
                    if pd.notna(price)
                    else None
                ),

            "currency":
                currency,

            "average_rating":
                (
                    float(rating)
                    if pd.notna(rating)
                    else None
                ),

            "review_count":
                (
                    int(review_count)
                    if pd.notna(
                        review_count
                    )
                    else None
                ),

            "semantic_score":
                semantic_score,

            "final_score":
                float(
                    final_score
                ),

            "faiss_row_id":
                int(row_id)

        })

    # ---------------------------------
    # 4. Sort results
    # ---------------------------------

    candidates.sort(
        key=lambda item:
            item["final_score"],
        reverse=True
    )

    return {
        "query": query,
        "requirements": requirements,
        "searched_vectors": search_k,
        "eligible_candidates": len(
            candidates
        ),
        "products": candidates[:top_k]
    }


print("PASS: DYNAMIC FAISS SEARCH READY")

PASS: DYNAMIC FAISS SEARCH READY


In [62]:
# CELL 52 — TEST DYNAMIC SEARCH

dynamic_result = (
    dynamic_semantic_search(
        query=test_query_dynamic,
        top_k=5,
        search_k=1000
    )
)

print("=" * 80)
print("DYNAMIC SEARCH RESULT")
print("=" * 80)

print(
    "QUERY:",
    dynamic_result["query"]
)

print(
    "\nREQUIREMENTS:",
    dynamic_result["requirements"]
)

print(
    "\nSEARCHED VECTORS:",
    dynamic_result["searched_vectors"]
)

print(
    "ELIGIBLE CANDIDATES:",
    dynamic_result["eligible_candidates"]
)

print("\nTOP PRODUCTS:")

for i, product in enumerate(
    dynamic_result["products"],
    start=1
):

    print(
        f"\n{i}.",
        product["product_name"]
    )

    print(
        "ID:",
        product["product_id"]
    )

    print(
        "PRICE:",
        product["price"],
        product["currency"]
    )

    print(
        "RATING:",
        product["average_rating"]
    )

    print(
        "SEMANTIC:",
        round(
            product["semantic_score"],
            4
        )
    )

    print(
        "FINAL SCORE:",
        round(
            product["final_score"],
            4
        )
    )

DYNAMIC SEARCH RESULT
QUERY: Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video và sử dụng cho điện thoại

REQUIREMENTS: {'original_query': 'Tôi cần thẻ nhớ microSD 32GB dưới 15 USD để lưu video và sử dụng cho điện thoại', 'capacity_gb': 32.0, 'max_price': 15.0, 'currency': 'USD'}

SEARCHED VECTORS: 1000
ELIGIBLE CANDIDATES: 129

TOP PRODUCTS:

1. SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card with Adapter - 98MB/s, C10, U1, Full HD, A1, Micro SD Card - SDSQUAR-032G-GN6MA
ID: amazon_B073JWXGNT
PRICE: 8.66 USD
RATING: 4.7
SEMANTIC: 0.7873
FINAL SCORE: 0.8232

2. SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-GN6MA)
ID: amazon_B011BRUOMO
PRICE: 8.99 USD
RATING: 4.6
SEMANTIC: 0.7799
FINAL SCORE: 0.8111

3. BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 microSD Memory Card for Dash Cam/Tablet/Phone (32GB U1)
ID: amazon_B0BZ4MFNY1
PRICE: 6.99 USD
RATING: 4.6
SEMANTIC: 0.7498
FINAL SCORE: 0.7768

4. SanDisk Ultra SDSQUNS-032G-GN3MN 32GB 80MB/s

In [63]:
# CELL 53 — PREPARE DYNAMIC REVIEW LOOKUP

import pandas as pd
import numpy as np

assert "product_id" in reviews.columns
assert "review_id" in reviews.columns
assert "review_text" in reviews.columns

reviews_dynamic = reviews.copy()

def usable_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null"
    }:
        return ""

    return text


def choose_review_text(row):

    smart_text = usable_text(
        row.get("smart_review_text")
    )

    original_text = usable_text(
        row.get("review_text")
    )

    return smart_text or original_text


reviews_dynamic["rag_review_text"] = (
    reviews_dynamic.apply(
        choose_review_text,
        axis=1
    )
)

reviews_dynamic = reviews_dynamic[
    reviews_dynamic[
        "rag_review_text"
    ].str.len() > 0
].copy()

reviews_by_product_dynamic = {
    product_id: group
    for product_id, group
    in reviews_dynamic.groupby(
        "product_id",
        sort=False
    )
}

print("PASS: DYNAMIC REVIEW LOOKUP READY")

print(
    "REVIEWS WITH TEXT:",
    len(reviews_dynamic)
)

print(
    "PRODUCTS WITH REVIEWS:",
    len(reviews_by_product_dynamic)
)

print("\nTOP 5 REVIEW AVAILABILITY:")

for product in dynamic_result["products"]:

    product_id = product["product_id"]

    group = reviews_by_product_dynamic.get(
        product_id
    )

    print(
        product_id,
        "|",
        0 if group is None else len(group),
        "reviews"
    )

PASS: DYNAMIC REVIEW LOOKUP READY
REVIEWS WITH TEXT: 766701
PRODUCTS WITH REVIEWS: 123304

TOP 5 REVIEW AVAILABILITY:
amazon_B073JWXGNT | 302 reviews
amazon_B011BRUOMO | 1354 reviews
amazon_B0BZ4MFNY1 | 8 reviews
amazon_B074B28H3S | 13 reviews
amazon_B016A5NH6C | 4 reviews


In [64]:
# CELL 54 — DYNAMIC REVIEW RETRIEVAL

import numpy as np
import pandas as pd


def retrieve_reviews_dynamic(
    product_id,
    query,
    max_reviews=5,
    candidate_limit=300
):

    group = reviews_by_product_dynamic.get(
        product_id
    )

    if group is None or group.empty:
        return []

    # Giới hạn số review cần embedding
    # để tránh xử lý quá nhiều mỗi lượt hỏi.
    candidates = group.head(
        candidate_limit
    ).copy()

    review_texts = (
        candidates[
            "rag_review_text"
        ]
        .astype(str)
        .tolist()
    )

    if not review_texts:
        return []

    query_vector = review_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    review_vectors = review_model.encode(
        review_texts,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    scores = (
        review_vectors
        @ query_vector[0]
    )

    candidates["relevance_score"] = (
        scores
    )

    selected = (
        candidates
        .sort_values(
            "relevance_score",
            ascending=False
        )
        .head(max_reviews)
    )

    result = []

    for _, row in selected.iterrows():

        rating = pd.to_numeric(
            row.get("rating"),
            errors="coerce"
        )

        result.append({
            "review_id": str(
                row["review_id"]
            ),

            "review_rating": (
                float(rating)
                if pd.notna(rating)
                else None
            ),

            "review_text": (
                str(
                    row["rag_review_text"]
                )[:500]
            ),

            "relevance_score": float(
                row["relevance_score"]
            )
        })

    return result


print("PASS: DYNAMIC REVIEW RETRIEVAL READY")

PASS: DYNAMIC REVIEW RETRIEVAL READY


In [65]:
# CELL 55 — BUILD DYNAMIC RAG CONTEXT

product_lookup_dynamic = (
    products
    .drop_duplicates("product_id")
    .set_index("product_id")
)


def clean_json_value(value):

    if value is None:
        return None

    if isinstance(value, (list, dict)):
        return value

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if hasattr(value, "item"):
        try:
            return value.item()
        except (AttributeError, ValueError):
            pass

    return value


def compact_text_dynamic(
    value,
    max_chars=400
):

    value = clean_json_value(
        value
    )

    if value is None:
        return ""

    text = str(value).strip()

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars]
        .rsplit(" ", 1)[0]
        + "..."
    )


def build_dynamic_rag_context(
    search_result,
    max_reviews=5
):

    query = search_result["query"]

    context_products = []

    for search_product in (
        search_result["products"]
    ):

        product_id = (
            search_product["product_id"]
        )

        assert (
            product_id
            in product_lookup_dynamic.index
        ), (
            f"Không tìm thấy metadata: "
            f"{product_id}"
        )

        metadata = (
            product_lookup_dynamic.loc[
                product_id
            ]
        )

        evidence = (
            retrieve_reviews_dynamic(
                product_id=product_id,
                query=query,
                max_reviews=max_reviews
            )
        )

        product_context = {

            "product_id":
                product_id,

            "title":
                search_product[
                    "product_name"
                ],

            "source":
                search_product[
                    "source"
                ],

            "price":
                search_product[
                    "price"
                ],

            "currency":
                search_product[
                    "currency"
                ],

            "product_average_rating":
                search_product[
                    "average_rating"
                ],

            "metadata_rating_count":
                search_product[
                    "review_count"
                ],

            "linked_review_count":
                clean_json_value(
                    metadata.get(
                        "etl_review_rows"
                    )
                ),

            "retrieved_evidence_count":
                len(evidence),

            "product_url":
                clean_json_value(
                    metadata.get(
                        "product_url"
                    )
                ),

            "description":
                compact_text_dynamic(
                    metadata.get(
                        "description"
                    ),
                    350
                ),

            "features":
                compact_text_dynamic(
                    metadata.get(
                        "product_features"
                    ),
                    450
                ),

            "reviews":
                evidence
        }

        context_products.append(
            product_context
        )

    return {

        "query":
            query,

        "requirements":
            search_result[
                "requirements"
            ],

        "searched_vectors":
            search_result[
                "searched_vectors"
            ],

        "eligible_candidates":
            search_result[
                "eligible_candidates"
            ],

        "products":
            context_products
    }


dynamic_rag_context = (
    build_dynamic_rag_context(
        dynamic_result,
        max_reviews=5
    )
)

print("PASS: DYNAMIC RAG CONTEXT BUILT")

print(
    "PRODUCTS:",
    len(
        dynamic_rag_context[
            "products"
        ]
    )
)

for i, product in enumerate(
    dynamic_rag_context["products"],
    start=1
):

    print(
        f"\n{i}.",
        product["product_id"]
    )

    print(
        "TITLE:",
        product["title"]
    )

    print(
        "PRICE:",
        product["price"],
        product["currency"]
    )

    print(
        "RATING:",
        product[
            "product_average_rating"
        ]
    )

    print(
        "LINKED REVIEWS:",
        product[
            "linked_review_count"
        ]
    )

    print(
        "RETRIEVED REVIEWS:",
        product[
            "retrieved_evidence_count"
        ]
    )

    print(
        "REVIEW IDS:",
        [
            r["review_id"]
            for r in product[
                "reviews"
            ]
        ]
    )

PASS: DYNAMIC RAG CONTEXT BUILT
PRODUCTS: 5

1. amazon_B073JWXGNT
TITLE: SanDisk 32GB Ultra MicroSDHC UHS-I Memory Card with Adapter - 98MB/s, C10, U1, Full HD, A1, Micro SD Card - SDSQUAR-032G-GN6MA
PRICE: 8.66 USD
RATING: 4.7
LINKED REVIEWS: 302
RETRIEVED REVIEWS: 5
REVIEW IDS: ['review_0529082', 'review_0445991', 'review_0426199', 'review_0561905', 'review_0639235']

2. amazon_B011BRUOMO
TITLE: SanDisk Ultra 32GB microSDHC UHS-I Card with Adapter, Silver, Standard Packaging (SDSQUNC-032G-GN6MA)
PRICE: 8.99 USD
RATING: 4.6
LINKED REVIEWS: 1354
RETRIEVED REVIEWS: 5
REVIEW IDS: ['review_0139542', 'review_0070564', 'review_0021589', 'review_0037177', 'review_0025295']

3. amazon_B0BZ4MFNY1
TITLE: BRAVEEAGLE FAT32 32GB Micro SD Card Class 10 microSD Memory Card for Dash Cam/Tablet/Phone (32GB U1)
PRICE: 6.99 USD
RATING: 4.6
LINKED REVIEWS: 8
RETRIEVED REVIEWS: 5
REVIEW IDS: ['review_0611660', 'review_0288655', 'review_0207438', 'review_0415732', 'review_0483465']

4. amazon_B074B28H3S
TI

In [66]:
# CELL 56 — GENERATE ANSWER FROM DYNAMIC RAG CONTEXT

import json
import time
import torch


DYNAMIC_SYSTEM_PROMPT = """
Bạn là chatbot tư vấn sản phẩm thương mại điện tử
sử dụng Retrieval-Augmented Generation (RAG).

Chỉ sử dụng thông tin trong RAG CONTEXT.

QUY TẮC:

1. Không tự tạo sản phẩm, giá, rating,
   thông số hoặc đường dẫn mua hàng.

2. Giá là giá ghi nhận trong dataset,
   không phải giá bán thời gian thực.

3. Giữ nguyên đơn vị tiền tệ.
   Không tự quy đổi USD sang VND.

4. product_average_rating là điểm đánh giá
   trung bình của sản phẩm trên thang 5.

5. metadata_rating_count là số lượt đánh giá
   trong metadata, không phải số review
   được hệ thống truy xuất.

6. linked_review_count là số review liên kết
   với sản phẩm trong dataset.

7. retrieved_evidence_count là số review
   được đưa vào RAG Context.

8. review_rating là điểm của một review cá nhân,
   không phải điểm trung bình sản phẩm.

9. Khi sử dụng nhận xét từ review,
   phải dẫn review_id tương ứng.

10. Không được tạo review_id không có
    trong RAG CONTEXT.

11. Tóm tắt review bằng tiếng Việt,
    không chép nguyên văn review dài.

12. Không khẳng định sản phẩm tương thích
    với thiết bị cụ thể khi chưa đủ thông tin.

13. Nếu review đề cập đến dung lượng khác,
    có thể sử dụng như ngữ cảnh so sánh,
    nhưng không được khẳng định đó là
    dung lượng của sản phẩm đang tư vấn.

14. Không khẳng định đã đọc toàn bộ review.

15. Nếu product_url thiếu, nói rằng dữ liệu
    chưa có đường dẫn mua hàng.

16. Không đưa ra kết luận tuyệt đối như
    "tốt nhất", "chắc chắn tương thích",
    "không bao giờ hỏng".

17. Chỉ trình bày sản phẩm thực sự có trong
    RAG CONTEXT.

18. Nếu dữ liệu không đủ để kết luận,
    phải nêu rõ giới hạn.

Trả lời bằng tiếng Việt, rõ ràng và dễ đọc.

CẤU TRÚC:

1. Tóm tắt nhu cầu người dùng.
2. Các sản phẩm phù hợp.
3. So sánh dựa trên dữ liệu.
4. Lưu ý về giá, khả năng tương thích
   và đường dẫn mua hàng.

Không bắt buộc giới thiệu đủ 5 sản phẩm.
""".strip()


def generate_dynamic_rag_answer(
    rag_context,
    max_new_tokens=1200
):

    query = rag_context["query"]

    context_json = json.dumps(
        rag_context,
        ensure_ascii=False,
        indent=2,
        default=str
    )

    user_prompt = f"""
CÂU HỎI NGƯỜI DÙNG:
{query}

RAG CONTEXT:
{context_json}

Hãy tư vấn dựa trên dữ liệu trên.

Phân biệt rõ rating trung bình của sản phẩm
với rating của từng review.

Mỗi nhận xét dựa trên review phải có review_id.

Không tự tạo giá, sản phẩm hoặc URL.
""".strip()

    messages = [
        {
            "role": "system",
            "content": DYNAMIC_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    model_inputs = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )
    )

    prompt_tokens = (
        model_inputs.shape[-1]
    )

    model_limit = (
        llm.config.max_position_embeddings
    )

    if (
        prompt_tokens + max_new_tokens
        > model_limit
    ):

        raise ValueError(
            "RAG Context vượt giới hạn "
            "token của Qwen."
        )

    start_time = time.time()

    with torch.inference_mode():

        outputs = llm.generate(
            input_ids=model_inputs.to(
                llm.device
            ),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0,
        model_inputs.shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    elapsed = (
        time.time() - start_time
    )

    return {
        "answer": answer,
        "prompt_tokens": int(
            prompt_tokens
        ),
        "generated_tokens": int(
            len(generated_tokens)
        ),
        "generation_seconds": round(
            elapsed,
            2
        )
    }


print("PASS: DYNAMIC QWEN GENERATOR READY")

PASS: DYNAMIC QWEN GENERATOR READY


In [67]:
# CELL 57 — DYNAMIC SEMANTIC RAG CHATBOT

import re


def audit_dynamic_answer(
    answer,
    rag_context
):

    valid_review_ids = {
        str(review["review_id"])
        for product in rag_context["products"]
        for review in product["reviews"]
        if review.get("review_id")
    }

    cited_review_ids = set(
        re.findall(
            r"review_\d+",
            answer
        )
    )

    invalid_review_ids = (
        cited_review_ids
        - valid_review_ids
    )

    return {
        "available_review_ids": len(
            valid_review_ids
        ),
        "cited_review_ids": sorted(
            cited_review_ids
        ),
        "invalid_review_ids": sorted(
            invalid_review_ids
        ),
        "citation_ids_valid": (
            len(invalid_review_ids) == 0
        )
    }


def chatbot(
    query,
    user_id=None,
    top_k=5,
    search_k=1000,
    max_reviews=5,
    max_new_tokens=1200
):

    if not isinstance(query, str):
        raise TypeError(
            "query phải là chuỗi."
        )

    query = query.strip()

    if not query:
        return {
            "query": query,
            "answer": (
                "Bạn muốn tìm sản phẩm gì? "
                "Hãy cho biết loại sản phẩm "
                "hoặc nhu cầu sử dụng."
            ),
            "products": [],
            "mode": "clarification"
        }

    # ---------------------------------
    # 1. Dynamic FAISS retrieval
    # ---------------------------------

    search_result = (
        dynamic_semantic_search(
            query=query,
            top_k=top_k,
            search_k=search_k
        )
    )

    if not search_result["products"]:

        return {
            "query": query,
            "answer": (
                "Mình chưa tìm thấy sản phẩm "
                "đáp ứng các điều kiện trong "
                "phạm vi dữ liệu đã tìm kiếm. "
                "Bạn có muốn điều chỉnh "
                "ngân sách hoặc dung lượng không?"
            ),
            "products": [],
            "requirements": search_result[
                "requirements"
            ],
            "mode": "no_match"
        }

    # ---------------------------------
    # 2. Dynamic review retrieval
    # ---------------------------------

    context = (
        build_dynamic_rag_context(
            search_result,
            max_reviews=max_reviews
        )
    )

    # ---------------------------------
    # 3. Qwen generation
    # ---------------------------------

    generation = (
        generate_dynamic_rag_answer(
            rag_context=context,
            max_new_tokens=max_new_tokens
        )
    )

    # ---------------------------------
    # 4. Citation audit
    # ---------------------------------

    audit = audit_dynamic_answer(
        generation["answer"],
        context
    )

    # ---------------------------------
    # 5. Return structured result
    # ---------------------------------

    return {
        "query": query,
        "user_id": user_id,
        "mode": "semantic_rag",
        "requirements": context[
            "requirements"
        ],
        "searched_vectors": context[
            "searched_vectors"
        ],
        "eligible_candidates": context[
            "eligible_candidates"
        ],
        "products": context[
            "products"
        ],
        "answer": generation[
            "answer"
        ],
        "generation": generation,
        "audit": audit
    }


print("PASS: DYNAMIC CHATBOT READY")

PASS: DYNAMIC CHATBOT READY


In [68]:
# CELL 58 — TEST CHATBOT WITH A NEW QUERY

new_query = (
    "Tôi cần thẻ nhớ microSD 64GB "
    "dưới 20 USD để lưu ảnh và video"
)

chatbot_result = chatbot(
    query=new_query,
    user_id=None,
    top_k=5,
    search_k=1000,
    max_reviews=5,
    max_new_tokens=1200
)

print("=" * 80)
print("CHATBOT TEST")
print("=" * 80)

print(
    "QUERY:",
    chatbot_result["query"]
)

print(
    "\nMODE:",
    chatbot_result["mode"]
)

print(
    "\nREQUIREMENTS:",
    chatbot_result.get(
        "requirements"
    )
)

print(
    "\nELIGIBLE CANDIDATES:",
    chatbot_result.get(
        "eligible_candidates"
    )
)

print("\nPRODUCTS:")

for i, product in enumerate(
    chatbot_result["products"],
    start=1
):

    print(
        f"{i}.",
        product["product_id"],
        "|",
        product["price"],
        product["currency"],
        "| rating:",
        product["product_average_rating"]
    )

print("\n" + "=" * 80)
print("CHATBOT ANSWER")
print("=" * 80)

print(
    chatbot_result["answer"]
)

print("\n" + "=" * 80)
print("AUDIT")
print("=" * 80)

print(
    chatbot_result.get(
        "audit"
    )
)

print("\nGENERATION:")
print(
    chatbot_result.get(
        "generation",
        {}
    ).get(
        "generation_seconds"
    ),
    "seconds"
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


CHATBOT TEST
QUERY: Tôi cần thẻ nhớ microSD 64GB dưới 20 USD để lưu ảnh và video

MODE: semantic_rag

REQUIREMENTS: {'original_query': 'Tôi cần thẻ nhớ microSD 64GB dưới 20 USD để lưu ảnh và video', 'capacity_gb': 64.0, 'max_price': 20.0, 'currency': 'USD'}

ELIGIBLE CANDIDATES: 96

PRODUCTS:
1. amazon_B0BRZPCQNJ | 16.99 USD | rating: 4.7
2. amazon_B09HKJY2QB | 6.49 USD | rating: 4.6
3. amazon_B0C1H1Q3C1 | 7.99 USD | rating: 4.6
4. amazon_B07LG5WBTS | 7.99 USD | rating: 4.6
5. amazon_B07XDCZ9J3 | 12.78 USD | rating: 4.7

CHATBOT ANSWER
### Sản phẩm phù hợp:

1. **SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter**
   - **Rating Trung Bình:** 4.7
   - **Review Đánh Giá:** 
     - **Review ID: review_0077221**
       - **Đánh Giá:** "I've purchased these memory cards from SanDisk before and they have worked well. My family gave me a drone (HS-710) for Christmas and it uses 32GB memory cards. These are fine for that sort of use. I'm satisfied with my purchase."
 

In [69]:
# CELL 59 — PRODUCT-LEVEL REVIEW AUDIT

from collections import defaultdict

review_owners = defaultdict(set)
review_text_by_owner = {}

for product in chatbot_result["products"]:

    product_id = product["product_id"]

    for review in product["reviews"]:

        review_id = str(
            review["review_id"]
        )

        review_owners[review_id].add(
            product_id
        )

        review_text_by_owner[
            (product_id, review_id)
        ] = review["review_text"]


print("=" * 80)
print("PRODUCT-LEVEL REVIEW AUDIT")
print("=" * 80)

for product in chatbot_result["products"]:

    print(
        "\nPRODUCT:",
        product["product_id"]
    )

    print(
        "TITLE:",
        product["title"]
    )

    print(
        "PRICE:",
        product["price"],
        product["currency"]
    )

    print(
        "REVIEW IDS:",
        [
            r["review_id"]
            for r in product["reviews"]
        ]
    )


print("\n" + "=" * 80)
print("CHECK REVIEW_0374815")
print("=" * 80)

target_review_id = "review_0374815"

owners = review_owners.get(
    target_review_id,
    set()
)

print("OWNERS:", owners)

for product_id in owners:

    print(
        "\nPRODUCT:",
        product_id
    )

    print(
        "TEXT:",
        review_text_by_owner[
            (product_id, target_review_id)
        ]
    )


print("\n" + "=" * 80)
print("DUPLICATE REVIEW IDS ACROSS PRODUCTS")
print("=" * 80)

duplicates = {
    review_id: sorted(owners)
    for review_id, owners
    in review_owners.items()
    if len(owners) > 1
}

print(duplicates)

PRODUCT-LEVEL REVIEW AUDIT

PRODUCT: amazon_B0BRZPCQNJ
TITLE: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
PRICE: 16.99 USD
REVIEW IDS: ['review_0126056', 'review_0077221', 'review_0008587', 'review_0476206', 'review_0253730']

PRODUCT: amazon_B09HKJY2QB
TITLE: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
PRICE: 6.49 USD
REVIEW IDS: ['review_0482507', 'review_0249331', 'review_0360039', 'review_0335997', 'review_0374815']

PRODUCT: amazon_B0C1H1Q3C1
TITLE: Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
PRICE: 7.99 USD
REVIEW IDS: ['review_0016087', 'review_0565323']

PRODUCT: amazon_B07LG5WBTS
TITLE: Micro Center 64GB Class 10 MicroSDXC Flash Memory Card with Adapter for Mobile Device Storage Phon

In [70]:
# CELL 60 — COMPACT CHATBOT PROMPT

COMPACT_SYSTEM_PROMPT = """
Bạn là chatbot tư vấn sản phẩm thương mại điện tử
sử dụng RAG.

Chỉ sử dụng dữ liệu được cung cấp.

QUY TẮC:

1. Chỉ giới thiệu sản phẩm có trong context.

2. Với mỗi sản phẩm, bắt buộc ghi:
   - Tên
   - Giá và đơn vị tiền tệ
   - Rating trung bình

3. Giá là giá ghi nhận trong dataset,
   không phải giá bán thời gian thực.

4. Mỗi nhận xét dựa trên review phải có
   review_id thuộc ĐÚNG sản phẩm đó.

5. Không gắn review của sản phẩm này
   cho sản phẩm khác.

6. Tóm tắt review bằng tiếng Việt.
   Không chép nguyên văn tiếng Anh.

7. Không suy diễn thông số hoặc
   khả năng tương thích.

8. Không tự tạo đường dẫn mua hàng.

9. Nếu sản phẩm là gói nhiều thẻ,
   phải giữ rõ thông tin số lượng thẻ.

10. Không khẳng định đã đọc toàn bộ review.

11. Không tự tạo sản phẩm 64GB đơn lẻ
    từ sản phẩm có tên 2x64GB.

TRẢ LỜI NGẮN GỌN:

- Tóm tắt nhu cầu trong một câu.
- Giới thiệu tối đa 3 sản phẩm.
- Mỗi sản phẩm tối đa 2 nhận xét review.
- Mỗi nhận xét phải có review_id.
- Kết thúc bằng lưu ý về giá lịch sử,
  khả năng tương thích và URL nếu thiếu.

Không cần lặp lại nguyên văn toàn bộ review.
""".strip()

print("PASS: COMPACT PROMPT READY")

PASS: COMPACT PROMPT READY


In [71]:
# CELL 61 — COMPACT DYNAMIC CHATBOT

import json
import time
import torch


def chatbot_compact(
    query,
    top_k=5,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=750
):

    # 1. FAISS retrieval

    search_result = dynamic_semantic_search(
        query=query,
        top_k=top_k,
        search_k=search_k
    )

    if not search_result["products"]:

        return {
            "query": query,
            "answer": (
                "Chưa tìm thấy sản phẩm đáp ứng "
                "các điều kiện trong phạm vi "
                "tìm kiếm hiện tại."
            ),
            "products": [],
            "mode": "no_match"
        }

    # 2. Review retrieval

    context = build_dynamic_rag_context(
        search_result,
        max_reviews=max_reviews
    )

    # 3. Chỉ đưa 3 sản phẩm vào LLM

    llm_products = context["products"][:3]

    compact_context = {
        "query": query,
        "requirements": context["requirements"],
        "products": llm_products
    }

    # 4. Generate

    messages = [
        {
            "role": "system",
            "content": COMPACT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "Câu hỏi:\n"
                + query
                + "\n\nDữ liệu:\n"
                + json.dumps(
                    compact_context,
                    ensure_ascii=False,
                    default=str
                )
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    assert (
        inputs.shape[-1] + max_new_tokens
        <= llm.config.max_position_embeddings
    )

    start = time.time()

    with torch.inference_mode():

        outputs = llm.generate(
            input_ids=inputs.to(
                llm.device
            ),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0,
        inputs.shape[-1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    elapsed = time.time() - start

    # Audit chỉ dùng review thuộc 3 sản phẩm
    # thực sự được cung cấp cho LLM.

    compact_audit = audit_dynamic_answer(
        answer,
        compact_context
    )

    return {
        "query": query,
        "mode": "semantic_rag_compact",
        "requirements": context["requirements"],
        "eligible_candidates": context[
            "eligible_candidates"
        ],
        "products": llm_products,
        "answer": answer,
        "audit": compact_audit,
        "prompt_tokens": int(
            inputs.shape[-1]
        ),
        "generated_tokens": int(
            len(generated)
        ),
        "generation_seconds": round(
            elapsed,
            2
        )
    }


print("PASS: COMPACT CHATBOT READY")

PASS: COMPACT CHATBOT READY


In [72]:
# CELL 62 — TEST COMPACT CHATBOT

compact_result = chatbot_compact(
    query=(
        "Tôi cần thẻ nhớ microSD 64GB "
        "dưới 20 USD để lưu ảnh và video"
    ),
    top_k=5,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=750
)

print("=" * 80)
print("CHATBOT COMPACT ANSWER")
print("=" * 80)

print(
    compact_result["answer"]
)

print("\n" + "=" * 80)
print("AUDIT")
print("=" * 80)

print(
    compact_result["audit"]
)

print(
    "\nPROMPT TOKENS:",
    compact_result["prompt_tokens"]
)

print(
    "GENERATED TOKENS:",
    compact_result["generated_tokens"]
)

print(
    "TIME:",
    compact_result["generation_seconds"],
    "seconds"
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


CHATBOT COMPACT ANSWER
- Tóm tắt nhu cầu: Thẻ nhớ microSD 64GB dưới 20 USD để lưu trữ hình ảnh và video.

1. **SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card**
   - **Giá:** 16.99 USD
   - **Review:** 
     - **Review ID:** review_0126056
     - **Tóm tắt:** Giá tốt hơn thẻ SD của Nintendo nhưng chỉ có 238GB chứ không phải 256GB.
     - **Đánh giá:** 4.0
   - **Khả năng tương thích:** Hiệu quả với các thiết bị Android như smartphone và máy tính bảng.

2. **Micro Center Premium 64GB microSDXC Card**
   - **Giá:** 6.49 USD
   - **Review:** 
     - **Review ID:** review_0482507
     - **Tóm tắt:** Sử dụng ổn định trong hơn 8 tháng, tốc độ nhanh và phù hợp cho việc sao lưu và phim.
     - **Đánh giá:** 5.0
   - **Khả năng tương thích:** Phù hợp cho các thiết bị như Reolink Lumus và PS Vita.

3. **Hugolog 64GB Micro SD Card**
   - **Giá:** 7.99 USD
   - **Review:** 
     - **Review ID:** review_0016087
     - **Tóm tắt:** Khối lượng không phù hợp khi vận chuyển, khó tìm thấy trong ba

In [73]:
# CELL 63 — LOCATE BPR FILES

from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

patterns = [
    "bpr_model.pkl",
    "cf_mappings.pkl",
    "step12_config.json"
]

for filename in patterns:

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    print(
        "\nFILE:",
        filename
    )

    if matches:

        for path in matches:
            print(
                "FOUND:",
                path
            )

    else:

        print(
            "NOT FOUND"
        )


FILE: bpr_model.pkl
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/bpr_model.pkl

FILE: cf_mappings.pkl
FOUND: /kaggle/input/datasets/trnquangtriu/bosung/cf_mappings.pkl

FILE: step12_config.json
FOUND: /kaggle/input/datasets/trnquangtriu/13dataset/step12_config.json


In [74]:
# CELL 64 — INSPECT STEP 12 CONFIG

import json
from pathlib import Path

config_files = list(
    Path("/kaggle/input").rglob(
        "step12_config.json"
    )
)

assert len(config_files) == 1, (
    f"Tìm thấy {len(config_files)} "
    "file step12_config.json."
)

with open(
    config_files[0],
    "r",
    encoding="utf-8"
) as f:

    step12_config = json.load(f)

print(
    json.dumps(
        step12_config,
        ensure_ascii=False,
        indent=2
    )
)

{
  "step": "step12_hybrid_product_recommendation",
  "status": "completed",
  "product_count": 164926,
  "faiss_vector_count": 164926,
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "candidate_k": 200,
  "top_k": 5,
  "semantic_ranking_weights": {
    "semantic": 0.75,
    "rating": 0.2,
    "review_count": 0.05
  },
  "hybrid_ranking": {
    "enabled": true,
    "method": "bounded_bpr_adjustment",
    "bpr_adjustment_coefficient": 0.04,
    "fallback": "semantic",
    "requires_known_user": true
  },
  "bpr": {
    "users": 26903,
    "items": 15546,
    "latent_dimension": 65,
    "product_coverage_percent": 9.43
  },
  "constraints": {
    "capacity": true,
    "budget": true,
    "currency": true,
    "automatic_currency_conversion": false
  },
  "review_rag_integrated": false,
  "knowledge_editing_integrated": false
}


In [75]:
# CELL 65 — LOAD BPR MODEL AND MAPPINGS

import pickle
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

bpr_files = list(
    INPUT_DIR.rglob(
        "bpr_model.pkl"
    )
)

mapping_files = list(
    INPUT_DIR.rglob(
        "cf_mappings.pkl"
    )
)

assert len(bpr_files) == 1, (
    f"BPR files: {bpr_files}"
)

assert len(mapping_files) == 1, (
    f"Mapping files: {mapping_files}"
)

with open(
    bpr_files[0],
    "rb"
) as f:

    bpr_model = pickle.load(f)

with open(
    mapping_files[0],
    "rb"
) as f:

    cf_mappings = pickle.load(f)

print("PASS: BPR LOADED")

print(
    "MODEL TYPE:",
    type(bpr_model).__name__
)

print(
    "USER FACTORS:",
    bpr_model.user_factors.shape
)

print(
    "ITEM FACTORS:",
    bpr_model.item_factors.shape
)

print("\nMAPPING KEYS:")

print(
    list(
        cf_mappings.keys()
    )
)

for key, value in cf_mappings.items():

    print(
        key,
        "|",
        type(value).__name__,
        "| length:",
        len(value)
    )

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


PASS: BPR LOADED
MODEL TYPE: BayesianPersonalizedRanking
USER FACTORS: (26903, 65)
ITEM FACTORS: (15546, 65)

MAPPING KEYS:
['user_to_idx', 'item_to_idx', 'idx_to_user', 'idx_to_item']
user_to_idx | dict | length: 26903
item_to_idx | dict | length: 15546
idx_to_user | ndarray | length: 26903
idx_to_item | ndarray | length: 15546


In [76]:
# CELL 66 — VALIDATE BPR MAPPINGS

import numpy as np

user_to_idx = cf_mappings["user_to_idx"]
item_to_idx = cf_mappings["item_to_idx"]

idx_to_user = cf_mappings["idx_to_user"]
idx_to_item = cf_mappings["idx_to_item"]

assert len(user_to_idx) == bpr_model.user_factors.shape[0]
assert len(item_to_idx) == bpr_model.item_factors.shape[0]

assert len(idx_to_user) == bpr_model.user_factors.shape[0]
assert len(idx_to_item) == bpr_model.item_factors.shape[0]

assert (
    bpr_model.user_factors.shape[1]
    == bpr_model.item_factors.shape[1]
)

# Kiểm tra mapping hai chiều
for idx in [
    0,
    len(idx_to_user) // 2,
    len(idx_to_user) - 1
]:
    user_id = str(idx_to_user[idx])

    assert int(user_to_idx[user_id]) == idx

for idx in [
    0,
    len(idx_to_item) // 2,
    len(idx_to_item) - 1
]:
    product_id = str(idx_to_item[idx])

    assert int(item_to_idx[product_id]) == idx

print("PASS: BPR MAPPINGS VALIDATED")

print(
    "USER FACTORS:",
    bpr_model.user_factors.shape
)

print(
    "ITEM FACTORS:",
    bpr_model.item_factors.shape
)

# Chọn một user có trong mô hình
test_bpr_user_id = str(idx_to_user[0])

print("\nTEST USER:", test_bpr_user_id)

PASS: BPR MAPPINGS VALIDATED
USER FACTORS: (26903, 65)
ITEM FACTORS: (15546, 65)

TEST USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA


In [77]:
# CELL 67 — CHECK BPR CANDIDATE COVERAGE

def check_bpr_coverage(search_result):

    products_found = search_result["products"]

    known_items = [
        product
        for product in products_found
        if product["product_id"] in item_to_idx
    ]

    total = len(products_found)

    return {
        "total_products": total,
        "bpr_known_products": len(known_items),
        "bpr_unknown_products": (
            total - len(known_items)
        ),
        "coverage_percent": (
            round(
                len(known_items) / total * 100,
                2
            )
            if total
            else 0
        ),
        "known_product_ids": [
            product["product_id"]
            for product in known_items
        ]
    }


coverage = check_bpr_coverage(
    dynamic_result
)

print("BPR COVERAGE FOR 32GB QUERY:")

for key, value in coverage.items():
    print(key, ":", value)


coverage_64gb = check_bpr_coverage(
    dynamic_semantic_search(
        query=(
            "Tôi cần thẻ nhớ microSD 64GB "
            "dưới 20 USD để lưu ảnh và video"
        ),
        top_k=20,
        search_k=1000
    )
)

print("\nBPR COVERAGE FOR 64GB QUERY:")

for key, value in coverage_64gb.items():
    print(key, ":", value)

BPR COVERAGE FOR 32GB QUERY:
total_products : 5
bpr_known_products : 4
bpr_unknown_products : 1
coverage_percent : 80.0
known_product_ids : ['amazon_B073JWXGNT', 'amazon_B011BRUOMO', 'amazon_B0BZ4MFNY1', 'amazon_B074B28H3S']

BPR COVERAGE FOR 64GB QUERY:
total_products : 20
bpr_known_products : 11
bpr_unknown_products : 9
coverage_percent : 55.0
known_product_ids : ['amazon_B0BRZPCQNJ', 'amazon_B09HKJY2QB', 'amazon_B07LG5WBTS', 'amazon_B07XDCZ9J3', 'amazon_B0BHJXCFXW', 'amazon_B0081EAK34', 'amazon_106171327X', 'amazon_B01LORO7BA', 'amazon_B09BP1T3TB', 'amazon_B01GR95HTQ', 'amazon_B07K7ZZJ21']


In [78]:
# CELL 68 — BOUNDED HYBRID RANKING

import numpy as np
import pandas as pd


BPR_ADJUSTMENT_COEFFICIENT = 0.04


def hybrid_rerank(
    search_result,
    user_id=None
):

    products_found = [
        dict(product)
        for product in search_result["products"]
    ]

    if not products_found:

        return {
            **search_result,
            "products": [],
            "ranking_mode": "no_match",
            "bpr_known_candidates": 0
        }

    # Giữ lại semantic score gốc
    for product in products_found:

        product["semantic_rank_score"] = float(
            product["final_score"]
        )

        product["hybrid_score"] = float(
            product["final_score"]
        )

        product["bpr_raw_score"] = None

        product["bpr_percentile"] = None

    # Không có user_id
    if user_id is None:

        return {
            **search_result,
            "products": products_found,
            "ranking_mode": "semantic_new_user",
            "bpr_known_candidates": 0
        }

    user_id = str(user_id)

    # User không có trong BPR
    if user_id not in user_to_idx:

        return {
            **search_result,
            "products": products_found,
            "ranking_mode": "semantic_unknown_user",
            "bpr_known_candidates": 0
        }

    user_idx = int(
        user_to_idx[user_id]
    )

    user_vector = (
        bpr_model.user_factors[
            user_idx
        ]
    )

    # Chỉ tính BPR cho sản phẩm đã biết
    known_positions = []

    raw_scores = []

    for position, product in enumerate(
        products_found
    ):

        product_id = str(
            product["product_id"]
        )

        if product_id not in item_to_idx:
            continue

        item_idx = int(
            item_to_idx[product_id]
        )

        item_vector = (
            bpr_model.item_factors[
                item_idx
            ]
        )

        score = float(
            np.dot(
                user_vector,
                item_vector
            )
        )

        if not np.isfinite(score):
            continue

        known_positions.append(
            position
        )

        raw_scores.append(
            score
        )

        product[
            "bpr_raw_score"
        ] = score

    # Cần >=2 sản phẩm có BPR để xếp percentile
    if len(raw_scores) < 2:

        return {
            **search_result,
            "products": products_found,
            "ranking_mode": "semantic_insufficient_bpr",
            "bpr_known_candidates": len(
                raw_scores
            )
        }

    # Percentile trong tập sản phẩm có BPR
    percentiles = (
        pd.Series(raw_scores)
        .rank(
            pct=True,
            method="average"
        )
        .to_numpy()
    )

    mean_percentile = float(
        percentiles.mean()
    )

    # Cộng điều chỉnh có giới hạn
    for position, percentile in zip(
        known_positions,
        percentiles
    ):

        adjustment = (
            BPR_ADJUSTMENT_COEFFICIENT
            * (
                float(percentile)
                - mean_percentile
            )
        )

        products_found[
            position
        ][
            "bpr_percentile"
        ] = float(
            percentile
        )

        products_found[
            position
        ][
            "hybrid_score"
        ] += adjustment

    products_found.sort(
        key=lambda product:
            product["hybrid_score"],
        reverse=True
    )

    return {
        **search_result,
        "products": products_found,
        "ranking_mode": "hybrid_bpr",
        "bpr_known_candidates": len(
            raw_scores
        )
    }


print("PASS: HYBRID RERANKER READY")

PASS: HYBRID RERANKER READY


In [79]:
# CELL 69 — TEST HYBRID RANKING

test_search = dynamic_semantic_search(
    query=(
        "Tôi cần thẻ nhớ microSD 64GB "
        "dưới 20 USD để lưu ảnh và video"
    ),
    top_k=20,
    search_k=1000
)

new_user_result = hybrid_rerank(
    test_search,
    user_id=None
)

known_user_result = hybrid_rerank(
    test_search,
    user_id=test_bpr_user_id
)

print("=" * 80)
print("NEW USER")
print("=" * 80)

print(
    "MODE:",
    new_user_result["ranking_mode"]
)

for i, p in enumerate(
    new_user_result["products"][:5],
    start=1
):

    print(
        i,
        p["product_id"],
        round(
            p["hybrid_score"],
            4
        )
    )


print("\n" + "=" * 80)
print("KNOWN USER")
print("=" * 80)

print(
    "USER:",
    test_bpr_user_id
)

print(
    "MODE:",
    known_user_result["ranking_mode"]
)

print(
    "BPR KNOWN CANDIDATES:",
    known_user_result[
        "bpr_known_candidates"
    ]
)

for i, p in enumerate(
    known_user_result["products"][:5],
    start=1
):

    print(
        i,
        p["product_id"],
        "| semantic:",
        round(
            p["semantic_rank_score"],
            4
        ),
        "| hybrid:",
        round(
            p["hybrid_score"],
            4
        ),
        "| BPR:",
        p["bpr_raw_score"]
    )

NEW USER
MODE: semantic_new_user
1 amazon_B0BRZPCQNJ 0.8106
2 amazon_B09HKJY2QB 0.7926
3 amazon_B0C1H1Q3C1 0.773
4 amazon_B07LG5WBTS 0.773
5 amazon_B07XDCZ9J3 0.7699

KNOWN USER
USER: amazon_AE22DOLTN2WN6B3TJ32UDXOEFNSA
MODE: hybrid_bpr
BPR KNOWN CANDIDATES: 11
1 amazon_B09HKJY2QB | semantic: 0.7926 | hybrid: 0.8035 | BPR: 0.057964518666267395
2 amazon_B0BRZPCQNJ | semantic: 0.8106 | hybrid: 0.7961 | BPR: -0.45570874214172363
3 amazon_B0081EAK34 | semantic: 0.7612 | hybrid: 0.7794 | BPR: 0.5653223395347595
4 amazon_106171327X | semantic: 0.7612 | hybrid: 0.7758 | BPR: 0.09034505486488342
5 amazon_B0C1H1Q3C1 | semantic: 0.773 | hybrid: 0.773 | BPR: None


In [80]:
# CELL 70 — DYNAMIC HYBRID PRODUCT SEARCH

def dynamic_hybrid_search(
    query,
    user_id=None,
    top_k=5,
    candidate_k=200,
    search_k=1000
):
    """
    Pipeline:
    Query
      -> FAISS
      -> Capacity / budget / currency filter
      -> Semantic ranking
      -> BPR reranking trên tập ứng viên rộng
      -> Final Top K
    """

    assert top_k >= 1
    assert candidate_k >= top_k
    assert search_k >= candidate_k

    # 1. Semantic search lấy tối đa 200 ứng viên
    semantic_result = dynamic_semantic_search(
        query=query,
        top_k=candidate_k,
        search_k=search_k
    )

    # 2. BPR reranking nếu user đã biết
    hybrid_result = hybrid_rerank(
        search_result=semantic_result,
        user_id=user_id
    )

    # 3. Chỉ chọn Top K sau khi rerank
    final_products = (
        hybrid_result["products"][:top_k]
    )

    return {
        **hybrid_result,
        "products": final_products,
        "user_id": user_id,
        "candidate_k": candidate_k,
        "top_k": top_k,
        "bpr_known_candidates_before_topk":
            hybrid_result["bpr_known_candidates"]
    }


print("PASS: DYNAMIC HYBRID SEARCH READY")

PASS: DYNAMIC HYBRID SEARCH READY


In [81]:
# CELL 71 — TEST FULL HYBRID SEARCH

hybrid_query = (
    "Tôi cần thẻ nhớ microSD 64GB "
    "dưới 20 USD để lưu ảnh và video"
)

hybrid_new_user = dynamic_hybrid_search(
    query=hybrid_query,
    user_id=None,
    top_k=5,
    candidate_k=200,
    search_k=1000
)

hybrid_known_user = dynamic_hybrid_search(
    query=hybrid_query,
    user_id=test_bpr_user_id,
    top_k=5,
    candidate_k=200,
    search_k=1000
)


def print_hybrid_result(result):

    print("=" * 75)

    print(
        "MODE:",
        result["ranking_mode"]
    )

    print(
        "USER:",
        result["user_id"]
    )

    print(
        "ELIGIBLE CANDIDATES:",
        result["eligible_candidates"]
    )

    print(
        "BPR KNOWN BEFORE TOP K:",
        result[
            "bpr_known_candidates_before_topk"
        ]
    )

    print("\nTOP PRODUCTS:")

    for rank, product in enumerate(
        result["products"],
        start=1
    ):

        print(
            f"\n{rank}.",
            product["product_id"]
        )

        print(
            "NAME:",
            product["product_name"]
        )

        print(
            "PRICE:",
            product["price"],
            product["currency"]
        )

        print(
            "SEMANTIC SCORE:",
            round(
                product["semantic_rank_score"],
                4
            )
        )

        print(
            "HYBRID SCORE:",
            round(
                product["hybrid_score"],
                4
            )
        )

        print(
            "BPR RAW:",
            product["bpr_raw_score"]
        )


print("\nNEW USER")
print_hybrid_result(
    hybrid_new_user
)

print("\nKNOWN USER")
print_hybrid_result(
    hybrid_known_user
)


NEW USER
MODE: semantic_new_user
USER: None
ELIGIBLE CANDIDATES: 96
BPR KNOWN BEFORE TOP K: 0

TOP PRODUCTS:

1. amazon_B0BRZPCQNJ
NAME: SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
PRICE: 16.99 USD
SEMANTIC SCORE: 0.8106
HYBRID SCORE: 0.8106
BPR RAW: None

2. amazon_B09HKJY2QB
NAME: Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
PRICE: 6.49 USD
SEMANTIC SCORE: 0.7926
HYBRID SCORE: 0.7926
BPR RAW: None

3. amazon_B0C1H1Q3C1
NAME: Hugolog 64GB Micro SD Card, Micro SDXC UHS-I Memory Card for LaView Camera– 95MB/s,633X,U3,C10, Full HD Video V30, A1, FAT32, High Speed Flash TF Card P500 for Phone/Tablet/PC/Computer
PRICE: 7.99 USD
SEMANTIC SCORE: 0.773
HYBRID SCORE: 0.773
BPR RAW: None

4. amazon_B07LG5WBTS
NAME: Micro Center 64GB Class 10 MicroSDXC Flash Memory Card with Adapter for Mobile Device Storage Phone, Tablet, Drone & Full HD V

In [82]:
# CELL 72 — HYBRID RAG CHATBOT

import json
import time
import torch


HYBRID_RAG_SYSTEM_PROMPT = """
Bạn là chatbot tư vấn sản phẩm thương mại điện tử.

Chỉ sử dụng thông tin trong RAG Context.

NHIỆM VỤ:
Viết phần tư vấn ngắn bằng tiếng Việt
cho các sản phẩm được cung cấp.

QUY TẮC:
1. Không tự tạo sản phẩm, giá, rating hoặc URL.
2. Không thay đổi thứ tự sản phẩm.
3. Không chép nguyên văn review tiếng Anh.
4. Khi nhận xét dựa trên review, ghi review_id.
5. Không gắn review của sản phẩm này cho sản phẩm khác.
6. Không khẳng định tương thích với thiết bị cụ thể
   nếu không đủ bằng chứng.
7. Không suy diễn chất lượng, độ bền hoặc tốc độ
   ngoài dữ liệu được cung cấp.
8. Nếu review có ý kiến tiêu cực, không biến thành
   nhận xét tích cực.
9. Giá trong dữ liệu không phải giá thời gian thực.

Chỉ viết:
- Nhận xét ngắn về từng sản phẩm.
- Lưu ý về nhu cầu sử dụng.

Không viết lại bảng giá hoặc rating.
Không cần giới thiệu đủ mọi sản phẩm.
""".strip()


def generate_hybrid_commentary(
    rag_context,
    max_new_tokens=450
):

    compact_context = {
        "query": rag_context["query"],
        "requirements":
            rag_context["requirements"],
        "products": []
    }

    for product in rag_context["products"]:

        compact_context["products"].append({
            "product_id":
                product["product_id"],

            "title":
                product["title"],

            "description":
                product["description"],

            "features":
                product["features"],

            "reviews":
                product["reviews"][:3]
        })

    messages = [
        {
            "role": "system",
            "content":
                HYBRID_RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(
                compact_context,
                ensure_ascii=False,
                default=str
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    assert (
        inputs.shape[-1] + max_new_tokens
        <= llm.config.max_position_embeddings
    )

    start = time.time()

    with torch.inference_mode():

        outputs = llm.generate(
            input_ids=inputs.to(
                llm.device
            ),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0,
        inputs.shape[-1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return {
        "text": answer,
        "prompt_tokens":
            int(inputs.shape[-1]),
        "generated_tokens":
            int(len(generated)),
        "generation_seconds":
            round(
                time.time() - start,
                2
            )
    }


def format_product_cards(
    rag_context
):

    lines = []

    for rank, product in enumerate(
        rag_context["products"],
        start=1
    ):

        price = product["price"]

        currency = product["currency"]

        rating = product[
            "product_average_rating"
        ]

        rating_count = product[
            "metadata_rating_count"
        ]

        lines.append(
            f"{rank}. {product['title']}"
        )

        lines.append(
            f"   Mã sản phẩm: "
            f"{product['product_id']}"
        )

        if price is not None:

            lines.append(
                f"   Giá trong dữ liệu: "
                f"{price} {currency}"
            )

        else:

            lines.append(
                "   Giá: Chưa có dữ liệu"
            )

        if rating is not None:

            lines.append(
                f"   Rating trung bình: "
                f"{rating}/5"
            )

        if rating_count is not None:

            lines.append(
                f"   Số lượt đánh giá "
                f"trong metadata: "
                f"{int(rating_count)}"
            )

        product_url = product.get(
            "product_url"
        )

        if product_url:

            lines.append(
                f"   URL: {product_url}"
            )

        lines.append("")

    return "\n".join(lines)


def chatbot_hybrid(
    query,
    user_id=None,
    top_k=3,
    candidate_k=200,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=450
):

    if not isinstance(query, str):
        raise TypeError(
            "query phải là chuỗi."
        )

    query = query.strip()

    if not query:

        return {
            "query": query,
            "mode": "clarification",
            "answer":
                "Bạn muốn tìm sản phẩm gì?",
            "products": []
        }

    # 1. FAISS + BPR
    search_result = dynamic_hybrid_search(
        query=query,
        user_id=user_id,
        top_k=top_k,
        candidate_k=candidate_k,
        search_k=search_k
    )

    if not search_result["products"]:

        return {
            "query": query,
            "mode": "no_match",
            "answer": (
                "Chưa tìm thấy sản phẩm đáp ứng "
                "điều kiện trong phạm vi "
                "tìm kiếm hiện tại."
            ),
            "products": []
        }

    # 2. Review RAG
    rag_context = build_dynamic_rag_context(
        search_result,
        max_reviews=max_reviews
    )

    # 3. Product cards từ dữ liệu
    product_cards = format_product_cards(
        rag_context
    )

    # 4. Qwen viết nhận xét
    generation = generate_hybrid_commentary(
        rag_context,
        max_new_tokens=max_new_tokens
    )

    # 5. Kiểm tra review_id
    audit = audit_dynamic_answer(
        generation["text"],
        rag_context
    )

    # 6. Câu trả lời hoàn chỉnh
    answer = (
        "SẢN PHẨM ĐƯỢC TÌM THẤY\n\n"
        + product_cards
        + "\nTƯ VẤN TỪ CHATBOT\n\n"
        + generation["text"]
        + "\n\nLưu ý: Giá là giá ghi nhận "
          "trong dataset, không phải "
          "giá bán thời gian thực."
    )

    return {
        "query": query,
        "user_id": user_id,
        "mode":
            search_result["ranking_mode"],
        "requirements":
            search_result["requirements"],
        "eligible_candidates":
            search_result["eligible_candidates"],
        "bpr_known_candidates":
            search_result[
                "bpr_known_candidates_before_topk"
            ],
        "products":
            rag_context["products"],
        "answer": answer,
        "generation": generation,
        "audit": audit
    }


print("PASS: HYBRID RAG CHATBOT READY")

PASS: HYBRID RAG CHATBOT READY


In [83]:
# CELL 73 — TEST PERSONALIZED HYBRID CHATBOT

personalized_result = chatbot_hybrid(
    query=(
        "Tôi cần thẻ nhớ microSD 64GB "
        "dưới 20 USD để lưu ảnh và video"
    ),
    user_id=test_bpr_user_id,
    top_k=3,
    candidate_k=200,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=450
)

print("=" * 80)
print("PERSONALIZED CHATBOT")
print("=" * 80)

print(
    "MODE:",
    personalized_result["mode"]
)

print(
    "BPR KNOWN CANDIDATES:",
    personalized_result[
        "bpr_known_candidates"
    ]
)

print(
    "\n",
    personalized_result["answer"]
)

print("\nAUDIT:")
print(
    personalized_result["audit"]
)

print("\nGENERATION:")
print(
    personalized_result["generation"]
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


PERSONALIZED CHATBOT
MODE: hybrid_bpr
BPR KNOWN CANDIDATES: 25

 SẢN PHẨM ĐƯỢC TÌM THẤY

1. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Mã sản phẩm: amazon_B09HKJY2QB
   Giá trong dữ liệu: 6.49 USD
   Rating trung bình: 4.6/5
   Số lượt đánh giá trong metadata: 15428

2. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Mã sản phẩm: amazon_B0BRZPCQNJ
   Giá trong dữ liệu: 16.99 USD
   Rating trung bình: 4.7/5
   Số lượt đánh giá trong metadata: 121509

3. SanDisk SDSDQUA-064G-A11 Professional Ultra 64GB MicroSDXC card is custom formatted for high speed, lossless recording! Includes Standard SD Adapter. (UHS-1 Class 10 Certified 30MB/sec) for GoPro HERO4 Black
   Mã sản phẩm: amazon_B0081EAK34
   Giá trong dữ liệu: 8.82 USD
   Rating trung bình: 4.6/5
   Số lượt đánh giá trong metadata: 2903

TƯ VẤN TỪ CHATBOT

### Sản phẩm 1: Amaz

In [84]:
# CELL 74 — DETERMINISTIC REVIEW EVIDENCE CARDS

def format_review_evidence_cards(
    rag_context,
    max_reviews_per_product=2,
    max_chars_per_review=260
):

    lines = []

    for rank, product in enumerate(
        rag_context["products"],
        start=1
    ):

        lines.append(
            f"SẢN PHẨM {rank}: "
            f"{product['product_id']}"
        )

        reviews_list = product.get(
            "reviews",
            []
        )

        if not reviews_list:

            lines.append(
                "Chưa có review văn bản "
                "được truy xuất."
            )

            lines.append("")
            continue

        for review in reviews_list[
            :max_reviews_per_product
        ]:

            review_id = str(
                review["review_id"]
            )

            review_text = str(
                review["review_text"]
            ).strip()

            if (
                len(review_text)
                > max_chars_per_review
            ):

                review_text = (
                    review_text[
                        :max_chars_per_review
                    ]
                    .rsplit(" ", 1)[0]
                    + "..."
                )

            lines.append(
                f"- [{review_id}] "
                f"{review_text}"
            )

        lines.append("")

    return "\n".join(lines)


review_cards = format_review_evidence_cards(
    {
        "products":
            personalized_result["products"]
    },
    max_reviews_per_product=2
)

print("=" * 80)
print("VERIFIED REVIEW EVIDENCE")
print("=" * 80)

print(review_cards)

VERIFIED REVIEW EVIDENCE
SẢN PHẨM 1: amazon_B09HKJY2QB
- [review_0482507] This MicroSD card works great.<br />I have used it for over 8 months now and it works great, it's actually really fast.<br />At first I was skeptical that a $30 256GB card was real, but I so far have about 50GB in one of them and all the data is there. It's...
- [review_0249331] I was looking for a microSD card that would fit in a SD2Vita adapter for my Playstation Vita. I've bought a variety of memory sticks from Micro Center in the past and of the 9 of them that I own, only one of sticks became a major problem and was faulty (slow...

SẢN PHẨM 2: amazon_B0BRZPCQNJ
- [review_0126056] Price was better than the Nintendo SD card but only had 238GB instead of 256GB.
- [review_0077221] I've purchased these memory cards from SanDisk before and they have worked well. My family gave me a drone (HS-710) for Christmas and it uses 32GB memory cards. These are fine for that sort of use. I'm satisfied with my purchase.

SẢN 

In [85]:
# CELL 75 — VERIFY PRODUCT–REVIEW RELATION

def verify_product_review_relation(
    rag_context
):

    errors = []

    total_reviews = 0

    for product in rag_context["products"]:

        product_id = str(
            product["product_id"]
        )

        valid_product_reviews = (
            reviews_by_product_dynamic.get(
                product_id
            )
        )

        if valid_product_reviews is None:

            if product["reviews"]:

                errors.append({
                    "product_id":
                        product_id,

                    "error":
                        "Product has no linked "
                        "reviews in lookup."
                })

            continue

        valid_review_ids = set(
            valid_product_reviews[
                "review_id"
            ].astype(str)
        )

        for review in product["reviews"]:

            total_reviews += 1

            review_id = str(
                review["review_id"]
            )

            if (
                review_id
                not in valid_review_ids
            ):

                errors.append({
                    "product_id":
                        product_id,

                    "review_id":
                        review_id,

                    "error":
                        "Review does not belong "
                        "to product."
                })

    return {
        "checked_reviews":
            total_reviews,

        "errors":
            errors,

        "passed":
            len(errors) == 0
    }


review_relation_audit = (
    verify_product_review_relation({
        "products":
            personalized_result[
                "products"
            ]
    })
)

print("PRODUCT–REVIEW AUDIT")

print(
    review_relation_audit
)

assert review_relation_audit[
    "passed"
], (
    "Phát hiện review không thuộc "
    "đúng sản phẩm."
)

print(
    "\nPASS: ALL RETRIEVED REVIEWS "
    "BELONG TO THEIR PRODUCTS"
)

PRODUCT–REVIEW AUDIT
{'checked_reviews': 9, 'errors': [], 'passed': True}

PASS: ALL RETRIEVED REVIEWS BELONG TO THEIR PRODUCTS


In [86]:
# CELL 76 — GROUNDED SHORT COMMENTARY

import json
import time
import torch


GROUNDED_SYSTEM_PROMPT = """
Bạn là trợ lý tư vấn sản phẩm.

Các thẻ sản phẩm đã được hệ thống tạo
bằng code. Bạn KHÔNG cần viết lại tên,
giá, rating hoặc thông số kỹ thuật.

Nhiệm vụ duy nhất:
Viết một đoạn tư vấn ngắn bằng tiếng Việt
dựa trên nhu cầu và bằng chứng review.

QUY TẮC:
- Không tự tạo thông số kỹ thuật.
- Không nêu tốc độ MB/s.
- Không tự tạo giá hoặc rating.
- Không khẳng định tương thích thiết bị.
- Không tự tạo review_id.
- Không gắn review của sản phẩm này
  cho sản phẩm khác.
- Không biến nhận xét tiêu cực
  thành nhận xét tích cực.
- Không chép nguyên văn review tiếng Anh.
- Không nói đã đọc toàn bộ review.

Nếu sử dụng nhận xét từ review,
phải ghi [review_id] ngay sau nhận xét.

Chỉ viết tối đa 3 câu ngắn.
Không sử dụng bảng.
Không viết tiêu đề.
Không giới thiệu lại danh sách sản phẩm.
""".strip()


def generate_grounded_commentary(
    query,
    rag_context,
    max_new_tokens=220
):

    evidence_context = {
        "query": query,
        "products": []
    }

    for product in rag_context[
        "products"
    ]:

        evidence_context[
            "products"
        ].append({

            "product_id":
                product["product_id"],

            "reviews":
                product["reviews"][:2]
        })

    messages = [
        {
            "role": "system",
            "content":
                GROUNDED_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(
                evidence_context,
                ensure_ascii=False,
                default=str
            )
        }
    ]

    inputs = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )
    )

    start = time.time()

    with torch.inference_mode():

        outputs = llm.generate(
            input_ids=inputs.to(
                llm.device
            ),
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0,
        inputs.shape[-1]:
    ]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return {
        "text": text,

        "prompt_tokens":
            int(inputs.shape[-1]),

        "generated_tokens":
            int(len(generated)),

        "generation_seconds":
            round(
                time.time() - start,
                2
            )
    }


print(
    "PASS: GROUNDED COMMENTARY "
    "GENERATOR READY"
)

PASS: GROUNDED COMMENTARY GENERATOR READY


In [87]:
# CELL 77 — TEST GROUNDED COMMENTARY

grounded_context = {
    "query":
        personalized_result["query"],

    "products":
        personalized_result["products"]
}

grounded_generation = (
    generate_grounded_commentary(
        query=personalized_result[
            "query"
        ],
        rag_context=grounded_context,
        max_new_tokens=220
    )
)

print("=" * 80)
print("GROUNDED COMMENTARY")
print("=" * 80)

print(
    grounded_generation["text"]
)

print("\n" + "=" * 80)
print("GENERATION INFO")
print("=" * 80)

print(
    grounded_generation
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


GROUNDED COMMENTARY
Sản phẩm Amazon B09HKJY2QB này rất đáng tin cậy. Người dùng trước đã đánh giá cao về hiệu suất của nó, đặc biệt là tốc độ đọc và ghi dữ liệu. Bạn có thể yên tâm rằng thẻ nhớ này sẽ đáp ứng yêu cầu lưu trữ hình ảnh và video của bạn mà không gặp bất kỳ vấn đề nào.

GENERATION INFO
{'text': 'Sản phẩm Amazon B09HKJY2QB này rất đáng tin cậy. Người dùng trước đã đánh giá cao về hiệu suất của nó, đặc biệt là tốc độ đọc và ghi dữ liệu. Bạn có thể yên tâm rằng thẻ nhớ này sẽ đáp ứng yêu cầu lưu trữ hình ảnh và video của bạn mà không gặp bất kỳ vấn đề nào.', 'prompt_tokens': 1000, 'generated_tokens': 77, 'generation_seconds': 5.91}


In [88]:
# CELL 78 — GROUNDED ANSWER QUALITY AUDIT

import re


OVERCONFIDENT_PATTERNS = [
    r"rất đáng tin cậy",
    r"hoàn toàn đáng tin cậy",
    r"bạn có thể yên tâm",
    r"không gặp bất kỳ vấn đề nào",
    r"chắc chắn đáp ứng",
    r"đảm bảo đáp ứng",
    r"hoàn toàn phù hợp",
    r"không bao giờ gặp lỗi",
]


REVIEW_CLAIM_PATTERNS = [
    r"người dùng.*đánh giá",
    r"người mua.*đánh giá",
    r"khách hàng.*nhận xét",
    r"theo.*review",
    r"phản hồi.*cho thấy",
]


def audit_grounded_commentary(
    text,
    rag_context
):

    available_review_ids = set()

    for product in rag_context["products"]:

        for review in product.get(
            "reviews", []
        ):

            available_review_ids.add(
                str(review["review_id"])
            )

    cited_review_ids = set(
        re.findall(
            r"review_\d+",
            text
        )
    )

    invalid_review_ids = (
        cited_review_ids
        - available_review_ids
    )

    overconfident_matches = [
        pattern
        for pattern in OVERCONFIDENT_PATTERNS
        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    ]

    review_claim_detected = any(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )
        for pattern in REVIEW_CLAIM_PATTERNS
    )

    missing_review_citation = (
        review_claim_detected
        and len(cited_review_ids) == 0
    )

    passed = (
        len(invalid_review_ids) == 0
        and len(overconfident_matches) == 0
        and not missing_review_citation
    )

    return {
        "passed": passed,

        "available_review_ids":
            len(available_review_ids),

        "cited_review_ids":
            sorted(cited_review_ids),

        "invalid_review_ids":
            sorted(invalid_review_ids),

        "overconfident_patterns":
            overconfident_matches,

        "review_claim_detected":
            review_claim_detected,

        "missing_review_citation":
            missing_review_citation
    }


commentary_audit = audit_grounded_commentary(
    grounded_generation["text"],
    grounded_context
)

print("GROUNDED COMMENTARY AUDIT")

for key, value in commentary_audit.items():
    print(key, ":", value)

GROUNDED COMMENTARY AUDIT
passed : False
available_review_ids : 9
cited_review_ids : []
invalid_review_ids : []
overconfident_patterns : ['rất đáng tin cậy', 'bạn có thể yên tâm', 'không gặp bất kỳ vấn đề nào']
review_claim_detected : True
missing_review_citation : True


In [89]:
# CELL 79 — SAFE FALLBACK COMMENTARY

def build_safe_fallback_commentary(
    rag_context
):

    products_list = rag_context.get(
        "products", []
    )

    if not products_list:

        return (
            "Chưa có đủ dữ liệu sản phẩm "
            "để đưa ra tư vấn."
        )

    lines = [
        (
            "Các sản phẩm trên được tìm thấy "
            "dựa trên nội dung tìm kiếm và "
            "các điều kiện lọc đã xác định."
        )
    ]

    for rank, product in enumerate(
        products_list,
        start=1
    ):

        product_id = product[
            "product_id"
        ]

        reviews_list = product.get(
            "reviews", []
        )

        if reviews_list:

            review_ids = [
                str(review["review_id"])
                for review in reviews_list[:2]
            ]

            lines.append(
                f"Sản phẩm {rank} "
                f"({product_id}) có "
                f"{len(reviews_list)} review "
                "được truy xuất trong "
                "RAG Context; ví dụ: "
                + ", ".join(
                    f"[{rid}]"
                    for rid in review_ids
                )
                + "."
            )

        else:

            lines.append(
                f"Sản phẩm {rank} "
                f"({product_id}) chưa có "
                "review văn bản được "
                "truy xuất trong "
                "RAG Context."
            )

    lines.append(
        "Các review là trải nghiệm "
        "của người mua trong dữ liệu, "
        "không bảo đảm sản phẩm tương thích "
        "với mọi thiết bị hoặc đáp ứng "
        "mọi nhu cầu sử dụng."
    )

    return "\n".join(lines)


fallback_commentary = (
    build_safe_fallback_commentary(
        grounded_context
    )
)

print("SAFE FALLBACK COMMENTARY")
print("=" * 70)

print(fallback_commentary)

SAFE FALLBACK COMMENTARY
Các sản phẩm trên được tìm thấy dựa trên nội dung tìm kiếm và các điều kiện lọc đã xác định.
Sản phẩm 1 (amazon_B09HKJY2QB) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0482507], [review_0249331].
Sản phẩm 2 (amazon_B0BRZPCQNJ) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0126056], [review_0077221].
Sản phẩm 3 (amazon_B0081EAK34) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0583903], [review_0344767].
Các review là trải nghiệm của người mua trong dữ liệu, không bảo đảm sản phẩm tương thích với mọi thiết bị hoặc đáp ứng mọi nhu cầu sử dụng.


In [90]:
# CELL 80 — SELECT VERIFIED COMMENTARY

def select_grounded_commentary(
    generation,
    rag_context
):

    audit = audit_grounded_commentary(
        generation["text"],
        rag_context
    )

    if audit["passed"]:

        return {
            "text":
                generation["text"],

            "source":
                "llm",

            "audit":
                audit
        }

    return {
        "text":
            build_safe_fallback_commentary(
                rag_context
            ),

        "source":
            "deterministic_fallback",

        "audit":
            audit
    }


selected_commentary = (
    select_grounded_commentary(
        grounded_generation,
        grounded_context
    )
)

print(
    "COMMENTARY SOURCE:",
    selected_commentary["source"]
)

print("\nFINAL COMMENTARY")
print("=" * 70)

print(
    selected_commentary["text"]
)

print("\nAUDIT")
print(
    selected_commentary["audit"]
)

COMMENTARY SOURCE: deterministic_fallback

FINAL COMMENTARY
Các sản phẩm trên được tìm thấy dựa trên nội dung tìm kiếm và các điều kiện lọc đã xác định.
Sản phẩm 1 (amazon_B09HKJY2QB) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0482507], [review_0249331].
Sản phẩm 2 (amazon_B0BRZPCQNJ) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0126056], [review_0077221].
Sản phẩm 3 (amazon_B0081EAK34) có 3 review được truy xuất trong RAG Context; ví dụ: [review_0583903], [review_0344767].
Các review là trải nghiệm của người mua trong dữ liệu, không bảo đảm sản phẩm tương thích với mọi thiết bị hoặc đáp ứng mọi nhu cầu sử dụng.

AUDIT
{'passed': False, 'available_review_ids': 9, 'cited_review_ids': [], 'invalid_review_ids': [], 'overconfident_patterns': ['rất đáng tin cậy', 'bạn có thể yên tâm', 'không gặp bất kỳ vấn đề nào'], 'review_claim_detected': True, 'missing_review_citation': True}


In [91]:
# CELL 81 — FINAL HYBRID RAG CHATBOT

def chatbot_final(
    query,
    user_id=None,
    top_k=3,
    candidate_k=200,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=220
):
    """
    Final Step 13 pipeline:

    User query
        -> FAISS retrieval
        -> Capacity / budget / currency filtering
        -> Semantic + optional BPR ranking
        -> Product review retrieval
        -> Deterministic product cards
        -> Qwen commentary
        -> Commentary audit
        -> Safe fallback when audit fails
    """

    if not isinstance(query, str):
        raise TypeError("query phải là chuỗi.")

    query = query.strip()

    if not query:
        return {
            "query": query,
            "user_id": user_id,
            "mode": "clarification",
            "answer": (
                "Bạn muốn tìm sản phẩm gì? "
                "Hãy mô tả loại sản phẩm, "
                "nhu cầu sử dụng hoặc ngân sách."
            ),
            "products": []
        }

    # ----------------------------------
    # 1. FAISS + optional BPR
    # ----------------------------------

    search_result = dynamic_hybrid_search(
        query=query,
        user_id=user_id,
        top_k=top_k,
        candidate_k=candidate_k,
        search_k=search_k
    )

    if not search_result["products"]:
        return {
            "query": query,
            "user_id": user_id,
            "mode": "no_match",
            "requirements": search_result[
                "requirements"
            ],
            "answer": (
                "Chưa tìm thấy sản phẩm đáp ứng "
                "điều kiện trong phạm vi tìm kiếm "
                "hiện tại. Bạn có thể điều chỉnh "
                "ngân sách hoặc yêu cầu sản phẩm."
            ),
            "products": []
        }

    # ----------------------------------
    # 2. Dynamic RAG context
    # ----------------------------------

    rag_context = build_dynamic_rag_context(
        search_result,
        max_reviews=max_reviews
    )

    # ----------------------------------
    # 3. Verify review ownership
    # ----------------------------------

    relation_audit = (
        verify_product_review_relation(
            rag_context
        )
    )

    if not relation_audit["passed"]:
        raise ValueError(
            "Review–product relation audit failed: "
            + str(relation_audit["errors"])
        )

    # ----------------------------------
    # 4. Deterministic product cards
    # ----------------------------------

    product_cards = format_product_cards(
        rag_context
    )

    # ----------------------------------
    # 5. Generate LLM draft
    # ----------------------------------

    generation = generate_grounded_commentary(
        query=query,
        rag_context=rag_context,
        max_new_tokens=max_new_tokens
    )

    # ----------------------------------
    # 6. Audit + fallback
    # ----------------------------------

    selected = select_grounded_commentary(
        generation,
        rag_context
    )

    # ----------------------------------
    # 7. Compose final answer
    # ----------------------------------

    answer = (
        "SẢN PHẨM ĐƯỢC TÌM THẤY\n\n"
        + product_cards
        + "\nTHÔNG TIN TƯ VẤN\n\n"
        + selected["text"]
        + "\n\nLưu ý: Giá được ghi nhận "
          "trong dataset, không phải giá "
          "bán thời gian thực. Dữ liệu hiện "
          "chưa có đường dẫn mua hàng cho "
          "các sản phẩm được hiển thị."
    )

    return {
        "query": query,
        "user_id": user_id,
        "mode": search_result[
            "ranking_mode"
        ],
        "requirements": rag_context[
            "requirements"
        ],
        "searched_vectors": search_result[
            "searched_vectors"
        ],
        "eligible_candidates": search_result[
            "eligible_candidates"
        ],
        "bpr_known_candidates": search_result[
            "bpr_known_candidates_before_topk"
        ],
        "products": rag_context[
            "products"
        ],
        "answer": answer,
        "commentary_source": selected[
            "source"
        ],
        "commentary_audit": selected[
            "audit"
        ],
        "review_relation_audit":
            relation_audit,
        "generation": generation
    }


print("PASS: FINAL HYBRID RAG CHATBOT READY")

PASS: FINAL HYBRID RAG CHATBOT READY


In [92]:
# CELL 82 — FINAL END-TO-END TEST

final_query = (
    "Tôi cần thẻ nhớ microSD 64GB "
    "dưới 20 USD để lưu ảnh và video"
)

final_result = chatbot_final(
    query=final_query,
    user_id=test_bpr_user_id,
    top_k=3,
    candidate_k=200,
    search_k=1000,
    max_reviews=3,
    max_new_tokens=220
)

print("=" * 80)
print("FINAL CHATBOT — STEP 13")
print("=" * 80)

print("QUERY:", final_result["query"])
print("MODE:", final_result["mode"])

print(
    "BPR KNOWN CANDIDATES:",
    final_result["bpr_known_candidates"]
)

print(
    "COMMENTARY SOURCE:",
    final_result["commentary_source"]
)

print(
    "REVIEW RELATION PASS:",
    final_result[
        "review_relation_audit"
    ]["passed"]
)

print("\n" + "=" * 80)
print("FINAL ANSWER")
print("=" * 80)

print(final_result["answer"])

print("\n" + "=" * 80)
print("COMMENTARY AUDIT")
print("=" * 80)

print(final_result["commentary_audit"])

print("\nGENERATION INFO:")
print({
    key: value
    for key, value
    in final_result["generation"].items()
    if key != "text"
})

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


FINAL CHATBOT — STEP 13
QUERY: Tôi cần thẻ nhớ microSD 64GB dưới 20 USD để lưu ảnh và video
MODE: hybrid_bpr
BPR KNOWN CANDIDATES: 25
COMMENTARY SOURCE: deterministic_fallback
REVIEW RELATION PASS: True

FINAL ANSWER
SẢN PHẨM ĐƯỢC TÌM THẤY

1. Micro Center Premium 64GB microSDXC Card, Nintendo-Switch Compatible Memory Card, UHS-I C10 U3 V30 4K UHD Video A1 Micro SD Card with Adapter (64GB)
   Mã sản phẩm: amazon_B09HKJY2QB
   Giá trong dữ liệu: 6.49 USD
   Rating trung bình: 4.6/5
   Số lượt đánh giá trong metadata: 15428

2. SanDisk 64GB 2-Pack Ultra microSDXC UHS-I Memory Card (2x64GB) with Adapter - SDSQUAB-064G-GN6MT
   Mã sản phẩm: amazon_B0BRZPCQNJ
   Giá trong dữ liệu: 16.99 USD
   Rating trung bình: 4.7/5
   Số lượt đánh giá trong metadata: 121509

3. SanDisk SDSDQUA-064G-A11 Professional Ultra 64GB MicroSDXC card is custom formatted for high speed, lossless recording! Includes Standard SD Adapter. (UHS-1 Class 10 Certified 30MB/sec) for GoPro HERO4 Black
   Mã sản phẩm: amazon

In [93]:
# CELL 83 — SAVE STEP 13 RESULTS

import json
from pathlib import Path
from datetime import datetime, timezone

STEP13_DIR = Path(
    "/kaggle/working/step13"
)

STEP13_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def save_json(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )


step13_config = {
    "step": "step13_hybrid_rag_llm",
    "status": "end_to_end_test_completed",
    "embedding_model": (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),
    "embedding_dimension": 384,
    "faiss_index_type": "IndexFlatIP",
    "llm": "Qwen/Qwen2.5-3B-Instruct",
    "ranking": {
        "semantic_weights": {
            "semantic": 0.75,
            "rating": 0.20,
            "review_count": 0.05
        },
        "bpr_adjustment_coefficient": 0.04,
        "candidate_k": 200,
        "fallback_for_unknown_user": "semantic"
    },
    "rag": {
        "review_retrieval": "MiniLM cosine similarity",
        "max_reviews_per_product": 3,
        "review_candidate_limit": 300
    },
    "generation_control": {
        "product_cards": "deterministic",
        "llm_commentary": True,
        "review_relation_audit": True,
        "rule_based_commentary_audit": True,
        "fallback": "deterministic_fallback"
    },
    "limitations": [
        "Prices are historical dataset values.",
        "Product URLs are unavailable in the dataset.",
        "BPR only covers a subset of products.",
        "Review retrieval considers at most 300 reviews per product.",
        "Rule-based audit does not guarantee factual correctness.",
        "Knowledge Editing is not integrated yet."
    ],
    "saved_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

config_path = (
    STEP13_DIR
    / "step13_config.json"
)

result_path = (
    STEP13_DIR
    / "step13_final_test.json"
)

save_json(
    step13_config,
    config_path
)

save_json(
    final_result,
    result_path
)

print("PASS: STEP 13 RESULTS SAVED")

print("CONFIG:", config_path)
print("TEST RESULT:", result_path)

print(
    "COMMENTARY SOURCE:",
    final_result["commentary_source"]
)

PASS: STEP 13 RESULTS SAVED
CONFIG: /kaggle/working/step13/step13_config.json
TEST RESULT: /kaggle/working/step13/step13_final_test.json
COMMENTARY SOURCE: deterministic_fallback


In [94]:
# CELL 84 — VERIFY STEP 13 SAVED ARTIFACTS

import json
from pathlib import Path

step13_dir = Path("/kaggle/working/step13")

required_files = [
    "step13_config.json",
    "step13_final_test.json"
]

for filename in required_files:

    path = step13_dir / filename

    assert path.exists(), (
        f"Missing file: {path}"
    )

    assert path.stat().st_size > 0, (
        f"Empty file: {path}"
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    print(
        "PASS:",
        filename,
        "| size:",
        path.stat().st_size,
        "bytes"
    )

with open(
    step13_dir / "step13_final_test.json",
    "r",
    encoding="utf-8"
) as f:

    saved_result = json.load(f)

assert saved_result["mode"] == "hybrid_bpr"

assert (
    saved_result["review_relation_audit"]["passed"]
    is True
)

assert (
    saved_result["commentary_source"]
    == "deterministic_fallback"
)

assert len(saved_result["products"]) == 3

print("\nPASS: STEP 13 ARTIFACTS VERIFIED")
print("MODE:", saved_result["mode"])
print("PRODUCTS:", len(saved_result["products"]))
print(
    "COMMENTARY SOURCE:",
    saved_result["commentary_source"]
)

PASS: step13_config.json | size: 1247 bytes
PASS: step13_final_test.json | size: 10684 bytes

PASS: STEP 13 ARTIFACTS VERIFIED
MODE: hybrid_bpr
PRODUCTS: 3
COMMENTARY SOURCE: deterministic_fallback
